# P04 · Notebook 30 · The kept system on DCASE 2025 Task 2, as it is and with attribute classes

**Where we stand.** The kept system (the frozen recipe fused with a second view) scored 70.63 % on DCASE 2022 Task 2, second beside that year's 31 teams (notebook 28). Notebook 29 showed that another scoring does not add to it. This is the last experiment of the program.

**What it does.** DCASE 2025 Task 2 is the latest challenge of this task with public ground truth. It is "first-shot": the eight machine types of the evaluation set are not in the development set, and every type has one section.
1. **Checks first.** The model files must be notebook 23's. The 23 zips are downloaded (MD5 against Zenodo) and every clip is checked.
2. **Part 1 · the frozen recipe, twice:** as it is (a type is a machine: 15 classes), and with one class per type and attribute string (AT). Six trainings of 10 epochs each.
3. **Part 2 · the second view,** unchanged: three trainings of 300 epochs.
4. **The development set** (seven types) chooses between the two systems by a rule: AT is the result only if it is at least 0.01 of the official score above the system as it is. A gate (G-DEV) stops the run only if a system looks broken.
5. **The evaluation set is scored once.** The scores of both systems and of their parts are written before any label is read. The ground truth is fetched only then, and the official evaluator is run.
6. **The reading, fixed in the rules:** the result's official score beside the challenge's 35 teams.

**Rules:** `results/mimii_beats/RULES_first_shot_2025_written_before_running.md`, written before this notebook existed.

**It needs** in Google Drive: `P04_BEATs`, as before. It writes to a new folder, `P04_FirstShot` (3 GB is enough).

**Runtime:**
- Choose **G4** (RTX PRO 6000): Runtime › Change runtime type. Then Runtime › Run all.
- About 4.7 GB of downloads from Zenodo (18,000 clips), twelve short trainings and three longer ones.
- **Time:** about 2 hours on G4, from the step times notebooks 27 and 28 measured there. Cells 10 and 14 print their own estimates for this GPU.
- Everything needed is saved to Drive. If Colab disconnects, run all again: finished trainings are skipped and an unfinished one continues from its last save.
- Once the evaluation set has been scored, a later run reads the saved results and scores nothing again.

**At the end** `p04_n30_results.zip` downloads. Send it back here.

In [ ]:
# 1 · Setup
import os, sys, json, time, hashlib, struct, zipfile, subprocess, shutil, urllib.request, re, math, gc, csv, base64, importlib
from pathlib import Path
from collections import Counter, deque
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Parameter
from sklearn.metrics import roc_auc_score
try:
    import torchaudio
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torchaudio"], check=True)
    import torchaudio

TEST = os.environ.get("P04_TEST") == "1"
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    MY = Path("/content/drive/MyDrive")
    OUT, WORK, CKPT_DIR = MY / "P04_FirstShot", Path("/content/work"), MY / "P04_BEATs"
else:
    OUT, WORK = Path(os.environ.get("P04_OUT", "P04_FirstShot")), Path(os.environ.get("P04_WORK", "work"))
    CKPT_DIR = Path(os.environ.get("P04_CKPT_DIR", "P04_BEATs"))
if not CKPT_DIR.exists():
    raise SystemExit(f"missing {CKPT_DIR}: the folder P04_BEATs (with BEATs_iter3_plus_AS2M.pt, as notebook 1 left it) must be in Google Drive › MyDrive")
for p in (OUT, WORK):
    p.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
AMP = DEVICE == "cuda"
GPU = torch.cuda.get_device_name(0) if DEVICE == "cuda" else ""
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9 if DEVICE == "cuda" else 0.0
if DEVICE != "cuda" and not TEST:
    raise SystemExit("No GPU. Runtime › Change runtime type › G4 (or another GPU), then Runtime › Run all.")
T_START = time.time()
DRIVE_FREE_GB = shutil.disk_usage(OUT).free / 1e9
print("device:", DEVICE, GPU, f"({GPU_GB:.0f} GB)" if GPU else "", "· torch", torch.__version__, "· torchaudio", torchaudio.__version__,
      f"· free local disk {shutil.disk_usage(WORK).free / 1e9:.0f} GB · free Drive {DRIVE_FREE_GB:.1f} GB")


def md5(path, chunk=8 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def sha256(path, chunk=8 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def download(url, dest):
    dest = Path(dest); part = dest.with_suffix(dest.suffix + ".part")
    try:
        if url.startswith("file://"):
            shutil.copy(url[7:], part)
        elif shutil.which("wget"):
            subprocess.run(["wget", "-q", "--show-progress", "-c", "-O", str(part), url], check=True)
        else:
            urllib.request.urlretrieve(url, part)
        part.rename(dest)
    finally:
        if part.exists() and not dest.exists():
            part.unlink()


download_one = download                              # notebook 11's single-connection download


def download(url, dest):
    # several connections per file with aria2c (notebook 19 saw about 2.4 MB/s per connection from Zenodo); falls back to one connection
    dest = Path(dest); part = dest.with_suffix(dest.suffix + ".part")
    if not url.startswith("file://"):
        if shutil.which("aria2c") is None and IN_COLAB:
            if subprocess.run(["apt-get", "-qq", "install", "-y", "aria2"], capture_output=True).returncode != 0:
                subprocess.run(["apt-get", "-qq", "update"], capture_output=True)
                subprocess.run(["apt-get", "-qq", "install", "-y", "aria2"], capture_output=True)
        if shutil.which("aria2c"):
            r = subprocess.run(["aria2c", "-x", "4", "-s", "4", "-k", "8M", "--file-allocation=none", "--max-tries=30", "--retry-wait=10",
                                "--allow-overwrite=true", "--auto-file-renaming=false", "--console-log-level=warn", "--summary-interval=0",
                                "--download-result=hide", "-d", str(dest.parent), "-o", part.name, url])
            if r.returncode == 0 and part.exists():
                part.rename(dest)
                return
            print(f"  aria2c failed for {dest.name} (code {r.returncode}); one connection instead", flush=True)
            for f in (part, Path(str(part) + ".aria2")):
                f.unlink(missing_ok=True)
    download_one(url, dest)

In [ ]:
# 2 · Every number below was fixed in the rules before this notebook ran
RULES = "results/mimii_beats/RULES_first_shot_2025_written_before_running.md"
# DCASE 2025 Task 2 on Zenodo: development, additional training and evaluation (MD5s from the record pages)
ZD, ZX, ZE = "https://zenodo.org/records/15097779/files/", "https://zenodo.org/records/15392814/files/", "https://zenodo.org/records/15519362/files/"
TYPES_DEV = ["ToyCar", "ToyTrain", "bearing", "fan", "gearbox", "slider", "valve"]
TYPES_EVAL = ["AutoTrash", "BandSealer", "CoffeeGrinder", "HomeCamera", "Polisher", "ScrewFeeder", "ToyPet", "ToyRCCar"]
DMD5 = {"ToyCar": "16bd2069065281ea7ac0e96894f8592c", "ToyTrain": "c2524ff503bc1631ca308dc840bc5010", "bearing": "b3a13693b69b477ce814db8b40783955",
        "fan": "61b6aae6d5ebb8cb937faa86741fbf47", "gearbox": "9542948d980fd575426f60f2a41060b8", "slider": "6c1f2d2ef0bc1a960db5938adb8425d1",
        "valve": "5640c0dc03982107ef354fe9480c9a83"}
XMD5 = {"AutoTrash": "0d89113bee9837402c609cb466c4a953", "BandSealer": "0e43610e140e34e0187b25f09d1f447b", "CoffeeGrinder": "9f39058361fde38fd5e24a52262d5977",
        "HomeCamera": "f81a89d772b1454af476059cb9800788", "Polisher": "09e62aea7f2a82b417e7c388c1060e92", "ScrewFeeder": "4a30f5aa52f019cac1d8653931ac4f44",
        "ToyPet": "0684d7185b4feea26b8479324f9c6573", "ToyRCCar": "74dc1d920983255d53b5214b0429b4d8"}
EMD5 = {"AutoTrash": "b33216913b71328d813bcbb3ec7415d7", "BandSealer": "78a582ca8092b35861542b1c34268952", "CoffeeGrinder": "359951de42956ef9a853dec6121a9517",
        "HomeCamera": "1b2e09f3283be0531fafeb5a66b58dd4", "Polisher": "7e50e4296cd7bd0ab94df37673df313e", "ScrewFeeder": "88035983d52753cf0bf2c9a692740222",
        "ToyPet": "c38c60b36b3cfdff5429fb6673eb9c57", "ToyRCCar": "1a6797436fe20d1a86224e65c8c19563"}
DC_DEV = {t_: (ZD + f"dev_{t_}.zip?download=1", f"dev_{t_}.zip", h) for t_, h in DMD5.items()}
XZIPS = {t_: (ZX + f"eval_data_{t_}_train.zip?download=1", f"eval_data_{t_}_train.zip", h) for t_, h in XMD5.items()}
DC_EVAL = {t_: (ZE + f"eval_data_{t_}_test.zip?download=1", f"eval_data_{t_}_test.zip", h) for t_, h in EMD5.items()}
# the evaluation set's ground truth and the official evaluator: one repository at one commit. EV_SHA is the SHA-256 of the script and its
# two tool files together; GT_SHA of the 24 ground-truth files together (each file's path, a zero byte and its content, in the order below)
EV_COMMIT = "f903124a40a4cdb3faf2b2d6fa69c6a9f1794192"
EV_ARCHIVE = (f"https://github.com/nttcslab/dcase2025_task2_evaluator/archive/{EV_COMMIT}.zip", "dcase2025_task2_evaluator.zip")
EV_SCRIPT, EV_SHA = "dcase2025_task2_evaluator.py", "b20577511228d4243264fb339c06f86d2766ec50334609a3a45c335cc3b11102"
EV_TOOLS = ["tools/plot_anm_score.py", "tools/test_plots.py"]
GT_PARTS = ("ground_truth_data", "ground_truth_domain", "ground_truth_attributes")   # labels, domains, and the name table the evaluator needs
GT_SHA = "e54a474722f1fb5d878bed257c75444251eeefae4e4dd40a0f5363f8cec68850"
SECTION = "section_00"                                                # every type has one section
SR, LDN_FLOOR = 16000, 1e-6
NS_RULE = 10 * SR                                                     # every system reads 10 s of a clip: the first 10 s, or a shorter clip repeated to fill them
NS_MIN = 1 * SR                                                       # gate G0(b): a clip is at least 1 s
# Part 1 · the frozen recipe: notebook 23's lines, unchanged (notebook 21's settings; the track list fixes the random streams)
TRACKS = ["ARR", "MIC+", "MIC"]
ALL_TRACKS = list(TRACKS) + ["DC"]                                            # fixed order for random seeds
CFG = dict(lora_rank=16, lora_alpha=32, lora_dropout=0.1, emb_dim=256, subcenters=3, arc_scale=30.0, arc_margin=0.2,
           mixup_prob=0.5, mixup_beta=0.5, crop_s=4.0, lr_lora=2e-4, lr_head=1e-3, wd_head=1e-4,
           batch=32, epochs=10, warmup_epochs=1, clip_norm=5.0, xc_cap=600, pl_k=4, pl_starts=10, pl_seed=0,
           ts_prob=0.5, pd_balance_batches=8, se_seeds=[0, 1, 2], embed_batch=32)        # notebook 21's, unchanged
ECHO_HF, ECHO_SHA_N15 = "yucongzh/echo-small-0824", "1cc306d581c97604d70e451d24e8616b517f4dff5b4c488a7ea1bc860fbdafcc"
B3_SHA_N22B = "ba0f139a46b3a22f4ff78873db92214146cddfaad12db7c026f6dd758df1be6a"
ENCODERS = {"BEATs": {"name": "BEATs iter3+ (AS2M)", "dim": 768, "layers": 12},
            "BEATs-iter3": {"name": "BEATs iter3", "dim": 768, "layers": 12, "file": "BEATs_iter3.pt",
                            "sed_file": "BEATs_iter3_from_PretrainedSED.pt",          # Amendment 1: the third source
                            "sed": "https://github.com/fschmid56/PretrainedSED/releases/download/v0.0.1/BEATs_ssl.pt",
                            "share": "https://1drv.ms/u/s!AqeByhGUtINrgcpxJUNDxg4eU0r-vA?e=qezPJ5",
                            "azure": ("https://valle.blob.core.windows.net/share/BEATs/BEATs_iter3.pt?sv=2020-08-04"
                                      "&st=2023-03-01T07%3A51%3A05Z&se=2033-03-02T07%3A51%3A00Z&sr=c&sp=rl&sig=QJXmSJG9DbMKf48UDIU1MfzIro8HQOf3sqlNXiflY1I%3D")},
            "EAT-base": {"name": "EAT-base (epoch 30, pre-trained)", "dim": 768, "layers": 12, "repo": "worstchan/EAT-base_epoch30_pretrain"},
            "EAT-large": {"name": "EAT-large (epoch 20, pre-trained)", "dim": 1024, "layers": 24, "repo": "worstchan/EAT-large_epoch20_pretrain"}}
NEW_ENCODERS = ["BEATs-iter3"]                                       # notebook 23 needs only BEATs iter3 (MIC+ and MIC)
ENC_ORDER = list(ENCODERS)
EAT_NORM, EAT_FRAMES = (-4.268, 4.569), 1024                         # EAT's model card
GB_RATIO = 1.5
FROZEN = [("BEATs", "B", [0, 1, 2]), ("BEATs-iter3", "B3", [0, 1, 2])]   # MIC's final system: encoder, file code, seeds
# Part 2 · the second view: notebook 25's lines, unchanged
REF_COMMIT = "674c759bfaa7723619a4b1d8d00eb9d155c50ba6"
SV = dict(n_fft=2048, hop=512, n_mels=128, win_len=1024, c_dim=128, arc_s=30.0, arc_m=0.7, alpha=0.5, lr=1e-4, eta_min_ratio=0.1,
          epochs=300, batch=64, split_ratio=0.95, seeds=[0, 1, 2], sine_floor=1e-9, save_every=10, embed_batch=64,
          gpu_headroom_gb=5.0)
REF_FRAMES, REF_KERNEL, REF_CLASSES = 313, (8, 20), 15                # the reference's sizes for 10 s; a type is a machine: 15 classes
CAL_MIN = 30                                                          # a spread of form ZP needs at least 30 clips per type
MIN_SOURCE_FIRST = 600                                                # gate G0(b): a type's first 600 training clips by name are source-domain clips
GDEV_BAR = 0.55                                                       # gate G-DEV, on the development set's source-domain line: a system at chance has about 0.50
GDEV_SYSTEMS = ["ZP"]                                                 # the gate reads the system as it is; its two parts are reported beside it
AS_IS = ["ZP", "frozen", "view_paper"]                                # the system as it is and its two parts
ADOPT_BAR = 0.01                                                      # the choice: attribute classes are the result from this much over the system as it is
# the yardsticks: the official score (%) of the best system of each of the challenge's 35 teams (the results page, read twice)
TEAMS = {'Wang_MYPS_task2_3': 61.62755928284949, 'Saengthong_SCITOK_task2_2': 61.569433614253896, 'Yang_NBU_task2_1': 61.20134568401735,
         'Fujimura_NU_task2_1': 59.99466314801132, 'Jiang_THUEE_task2_2': 59.79329753333621, 'Zheng_SJTU-AITHU_task2_2': 59.49972264254595,
         'Guan_HEU_task2_1': 58.25344060002535, 'Ozeki_MELCO_task2_1': 58.23359688192865, 'Zhou_XAUAT_task2_1': 58.1881454965969,
         'Huang_XJU_task2_1': 58.14052430128241, 'Fu_CUMT_task2_4': 58.13668049404887, 'Qian_nivic_task2_2': 58.01974104378345,
         'Kim_AISTAT_task2_4': 57.95495072758183, 'Zhong_USTC_task2_4': 57.874646573614505, 'Wang_UniS_task2_1': 57.754303274825524,
         'Cai_NCUT_task2_3': 57.5585903081211, 'WT_IACAS_task2_2': 57.44073696264025, 'CHUNG_KUCAU_task2_4': 57.227161466047036,
         'Wang_ZJU_task2_4': 56.866003688314095, 'Sera_TMU_task2_1': 56.567561042877735, 'Ding_HFUU_task2_4': 56.37774443771024,
         'Zhao_CUMT_task2_3': 56.05710040362136, 'Zhang_DKU_task2_4': 55.76153468345078, 'Zhang_NWPU_task2_2': 55.42463764307989,
         'Lin_IASP_task2_4': 54.776114357402165, 'Li_XJTLU_task2_2': 53.469426047012206, 'Zhou_XJU_task2_4': 52.90746388052138,
         'Lobanov_ITMO_task2_2': 52.52595001212435, 'Bian_TGU_task2_3': 51.417480569726926, 'Chao_BUCT_task2_3': 50.13154697532129,
         'Vijayyan_SNUC_task2_1': 49.90505686153821, 'Kim_DAU_task2_2': 48.96578012474582, 'Dung_CNTT1PTIT_task2_1': 48.71892564275941,
         'Kret_CU_task2_1': 47.90299388774425, 'Emon_HDK_task2_1': 45.15099353175931}
BASELINES = {"DCASE2025_baseline_task2_MAHALA": 56.50558189601554, "DCASE2025_baseline_task2_MSE": 54.42827820713174}
# the result's words, on scores in % of two decimals: above the first team; at least the 3rd team's; at least the 18th team's (the middle of 35);
# at least the better baseline's
Q1_FIRST, Q1_LEADING, Q1_UPPER, Q1_BASE = 61.63, 61.20, 57.23, 56.51
Q3_BAR = 0.01                                                         # the fusion's words, on the official score
AE_EVAL_SHIFT = (71.99, 47.88)                                        # the Mahalanobis baseline on the evaluation set: harmonic mean AUC (source), AUC (target)
# the baselines' development tables (the task page): per type AUC (source), AUC (target), pAUC, %
PUBLISHED_DEV = {"Autoencoder baseline (MSE)": {"ToyCar": (71.05, 53.52, 49.7), "ToyTrain": (61.76, 56.46, 50.19), "bearing": (66.53, 53.15, 61.12),
                                                "fan": (70.96, 38.75, 49.46), "gearbox": (64.8, 50.49, 52.49), "slider": (70.1, 48.77, 52.32),
                                                "valve": (63.53, 67.18, 57.35)},
                 "Autoencoder baseline (Mahalanobis)": {"ToyCar": (73.17, 50.91, 49.05), "ToyTrain": (50.87, 46.15, 48.32), "bearing": (63.63, 59.03, 61.86),
                                                        "fan": (77.99, 38.56, 50.82), "gearbox": (73.26, 51.61, 55.07), "slider": (73.79, 50.27, 53.61),
                                                        "valve": (56.22, 61.0, 52.53)}}
NAME = {t_: t_ for t_ in TYPES_DEV + TYPES_EVAL}; NAME["slider"] = "slide rail"
EVAL_TOL, DRIVE_MIN_GB, DISK_MIN_GB = 1e-9, 3.0, 25.0
if TEST:
    _dc = json.loads(os.environ["P04_TEST_DCASE"])
    _c = json.loads(os.environ.get("P04_TEST_CFG", "{}"))
    TYPES_DEV, TYPES_EVAL = list(_dc["dev"]), list(_dc["add"])        # the synthetic sets hold fewer machine types
    DC_DEV, XZIPS, DC_EVAL = ({t_: tuple(v) for t_, v in _dc[k_].items()} for k_ in ("dev", "add", "eval"))
    EV_ARCHIVE, EV_SHA, GT_SHA = tuple(_dc["evaluator_archive"]), _dc["evaluator_sha256"], _dc["ground_truth_sha256"]
    CFG.update(_c.get("cfg", {})); SV.update(_c.get("sv", {}))
    for k_, v_ in _c.get("encoders", {}).items():
        ENCODERS[k_].update(v_)
    CAL_MIN, NS_RULE, MIN_SOURCE_FIRST = _c.get("cal_min", CAL_MIN), _c.get("clip_samples", NS_RULE), _c.get("min_source_first", MIN_SOURCE_FIRST)
    GDEV_BAR, REF_CLASSES = _c.get("gdev_bar", GDEV_BAR), len(TYPES_DEV) + len(TYPES_EVAL)
TYPES = TYPES_DEV + TYPES_EVAL
DEV_IDS = {t_: ([SECTION] if t_ in TYPES_DEV else []) for t_ in TYPES}   # the sections with labelled test clips
EVAL_IDS = {t_: ([SECTION] if t_ in TYPES_EVAL else []) for t_ in TYPES}  # the sections with unlabelled test clips
assert sorted(DC_DEV) == sorted(TYPES_DEV) and sorted(XZIPS) == sorted(DC_EVAL) == sorted(TYPES_EVAL) and not set(TYPES_DEV) & set(TYPES_EVAL)
FROZEN_CAP = CFG["xc_cap"]                                            # the frozen recipe trains on the first 600 training clips of a type
assert [sd for _, _, sd in FROZEN] == [CFG["se_seeds"]] * 2 and ENC_ORDER[:2] == [e for e, _, _ in FROZEN]
assert len(TEAMS) == 35 and sorted(TEAMS.values(), reverse=True) == list(TEAMS.values()), "the team ranking is not in order"
_ts = [round(v, 2) for v in TEAMS.values()]
assert (_ts[0], _ts[2], _ts[17], round(BASELINES["DCASE2025_baseline_task2_MAHALA"], 2)) == (Q1_FIRST, Q1_LEADING, Q1_UPPER, Q1_BASE), "the words' lines are not the table's"
print("rules:", RULES, "·", json.dumps(CFG), "·", json.dumps(SV))
if not TEST and DRIVE_FREE_GB < DRIVE_MIN_GB:
    raise SystemExit(f"stop: Drive has {DRIVE_FREE_GB:.1f} GB free and this run needs {DRIVE_MIN_GB} GB. Free some space in Google Drive, then run again.")
if not TEST and shutil.disk_usage(WORK).free / 1e9 < DISK_MIN_GB:
    raise SystemExit(f"stop: the local disk has {shutil.disk_usage(WORK).free / 1e9:.0f} GB free and this run needs {DISK_MIN_GB:.0f} GB")

In [ ]:
# 3 · The official BEATs code (notebook 23's cell, unchanged: the same files as notebook 1)
BEATS_RAW = "https://raw.githubusercontent.com/microsoft/unilm/master/beats/"
CODE_SHA256 = {"BEATs.py": "27f289db7c56ce26f2ceb50d3719854b91b2dec1c2830d8b1dd8de1bbee19eeb",
               "backbone.py": "31c0378379a7e0f1d1069f9da444fb86890fe1ea078959a2dcd39640cdcadbaa",
               "modules.py": "edeb6b6cd6a784da749f932c3e0783c0bce556fc768d0d23a4d53d4b819eb424"}
CODE_DIR = WORK / "beats"; CODE_DIR.mkdir(exist_ok=True)
code_sha = {}
for f, want in CODE_SHA256.items():
    p = CODE_DIR / f
    if not p.exists():
        download(BEATS_RAW + f, p)
    code_sha[f] = sha256(p)
    print(f, "same as notebook 1" if code_sha[f] == want else f"CHANGED upstream ({code_sha[f][:12]}…)")
try:
    import torchaudio.compliance.kaldi
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torchaudio"], check=True)
sys.path.insert(0, str(CODE_DIR))
from BEATs import BEATs, BEATsConfig

In [ ]:
# 4 · The checkpoint notebook 1 used (from Drive › P04_BEATs); its SHA-256 must match
CKPT_NAME, CKPT_SHA_N1 = "BEATs_iter3_plus_AS2M.pt", "d43cbfad4d7b56381c061d7a24774f908d4d94c72961f6eb1d9090ff18cd8d34"
ckpt = CKPT_DIR / CKPT_NAME
if not ckpt.exists():
    raise SystemExit(f"Put {CKPT_NAME} in Google Drive › MyDrive › P04_BEATs (notebook 1 left it there), then run again.")
CKPT_SHA = sha256(ckpt)
print(CKPT_NAME, "· same file as notebook 1:", CKPT_SHA == CKPT_SHA_N1)
if not TEST:
    assert CKPT_SHA == CKPT_SHA_N1, "stop: not the checkpoint notebook 1 used"
try:
    chk = torch.load(ckpt, map_location="cpu", weights_only=True)
except Exception:
    chk = torch.load(ckpt, map_location="cpu", weights_only=False)
cfg = BEATsConfig(chk["cfg"])
model = BEATs(cfg); model.load_state_dict(chk["model"]); model.eval().to(DEVICE)
print("layers:", cfg.encoder_layers, "· fine-tuned:", cfg.finetuned_model)

In [ ]:
# 5 · Model pieces (notebook 23's cell, unchanged; notebook 2's code): our forward pass, LoRA, the head and the sub-center ArcFace layer
def encode(x, amp):
    """Waveforms (B, N) in [-1, 1] → mean over time of the last layer (B, 768), float32.
    Same steps as BEATs.extract_features without padding; the filter banks are always computed in float32."""
    with torch.autocast(device_type=DEVICE, enabled=False):
        fb = model.preprocess(x.float())                              # (B, frames, 128)
    with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=amp):
        f = model.patch_embedding(fb.unsqueeze(1))
        f = f.reshape(f.shape[0], f.shape[1], -1).transpose(1, 2)
        f = model.layer_norm(f)
        if model.post_extract_proj is not None:
            f = model.post_extract_proj(f)
        h, _ = model.encoder(model.dropout_input(f), padding_mask=None)
    return h.float().mean(1)


class LoRALinear(nn.Module):
    """y = W x + (α / r) · B A x. W stays frozen; B starts at zero, so the model starts exactly as the frozen one."""
    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        self.base, self.r, self.scale = base, r, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features, device=base.weight.device))
        self.B = nn.Parameter(torch.empty(base.out_features, r, device=base.weight.device))
        self.drop = nn.Dropout(dropout)
        self.reset()

    def reset(self):
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        nn.init.zeros_(self.B)

    @property
    def weight(self):
        return self.base.weight

    @property
    def bias(self):
        return self.base.bias

    def forward(self, x):
        return self.base(x) + F.linear(F.linear(self.drop(x), self.A), self.B) * self.scale


class Head(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.lin = nn.Linear(d_in, d_out, bias=False)
        self.bn = nn.BatchNorm1d(d_out)

    def forward(self, x):
        return self.bn(self.lin(x))


class SubCenterArcFace(nn.Module):
    """Cosine to the closest of k centres per class; additive angular margin m on the true class; scale s."""
    def __init__(self, d, n_classes, k, s, m):
        super().__init__()
        self.W = nn.Parameter(torch.empty(n_classes * k, d))
        nn.init.xavier_uniform_(self.W)
        self.n, self.k, self.s, self.m = n_classes, k, s, m
        self.cos_m, self.sin_m = math.cos(m), math.sin(m)
        self.th, self.mm = math.cos(math.pi - m), math.sin(math.pi - m) * m

    def cosine(self, e):
        c = F.linear(F.normalize(e.float(), dim=1), F.normalize(self.W.float(), dim=1))
        return c.view(-1, self.n, self.k).max(dim=2).values

    def loss(self, cos, y):
        sin = torch.sqrt((1.0 - cos * cos).clamp(0.0, 1.0))
        phi = cos * self.cos_m - sin * self.sin_m
        phi = torch.where(cos > self.th, phi, cos - self.mm)      # keeps the target logit monotonic near θ = π
        one = F.one_hot(y, self.n).to(cos.dtype)
        return F.cross_entropy(self.s * (one * phi + (1.0 - one) * cos), y)


def equal_length_batches(clips, idx, batch):
    """Consecutive clips of the same length, at most `batch` per group (the same grouping as notebook 1)."""
    buf = []
    for i in idx:
        if buf and len(clips[i]) != len(clips[buf[0]]):
            yield buf; buf = []
        buf.append(i)
        if len(buf) == batch:
            yield buf; buf = []
    if buf:
        yield buf


def as_float(clips, ids):
    return torch.from_numpy(np.stack([clips[i] for i in ids]).astype(np.float32) / 32768.0).to(DEVICE)

In [ ]:
# 6 · LoRA on the query and value projections of all 12 layers, with one switch (PD's frozen teacher runs with it off); checks; a timed step
for p_ in model.parameters():
    p_.requires_grad_(False)                                          # the original weights never change


class LoRASwitch(LoRALinear):
    on = True                                                         # one switch for every block

    def forward(self, x):
        y = self.base(x)
        return y + F.linear(F.linear(self.drop(x), self.A), self.B) * self.scale if LoRASwitch.on else y


def tokens(x, amp):
    # waveforms (B, N) in [-1, 1] → last-layer tokens (B, T, 768), float32: notebook 2's encode without the final mean
    with torch.autocast(device_type=DEVICE, enabled=False):
        fb = model.preprocess(x.float())                              # filter banks always in float32
    with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=amp):
        f = model.patch_embedding(fb.unsqueeze(1))
        f = f.reshape(f.shape[0], f.shape[1], -1).transpose(1, 2)
        f = model.layer_norm(f)
        if model.post_extract_proj is not None:
            f = model.post_extract_proj(f)
        h, _ = model.encoder(model.dropout_input(f), padding_mask=None)
    return h.float()


rng0 = np.random.default_rng(0)
CROP = int(CFG["crop_s"] * SR)
x4 = torch.from_numpy(rng0.uniform(-0.1, 0.1, (4, CROP)).astype(np.float32)).to(DEVICE)
with torch.no_grad():
    Z4 = tokens(x4, amp=False).mean(1)
    ref = model.extract_features(x4)[0].mean(1)
    gap_e = float((encode(x4, amp=False) - Z4).abs().max())
gap = float((Z4 - ref).abs().max())
print(f"our forward pass against BEATs.extract_features: largest difference {gap:.2e} (notebook 2's encode: {gap_e:.1e})")
assert gap < 1e-3 and gap_e < 1e-5, "stop: our forward pass does not reproduce BEATs"
LORA = []
for layer in model.encoder.layers:
    att = layer.self_attn
    for name in ("q_proj", "v_proj"):
        w = LoRASwitch(getattr(att, name), CFG["lora_rank"], CFG["lora_alpha"], CFG["lora_dropout"])
        setattr(att, name, w); LORA.append(w)
with torch.no_grad():
    gap = float((tokens(x4, amp=False).mean(1) - Z4).abs().max())
print(len(LORA), "LoRA blocks · with LoRA at zero, largest difference from frozen BEATs:", f"{gap:.2e}")
assert gap < 1e-5, "stop: adding LoRA changed the model before training"


def lora_params():
    return [p for w in LORA for p in (w.A, w.B)]


def fresh_adapters(seed, n_classes):
    # LoRA back to zero, a new head and a new ArcFace layer: every training starts from the same frozen BEATs
    torch.manual_seed(seed)
    for w in LORA:
        w.reset()
    head = Head(768, CFG["emb_dim"]).to(DEVICE)
    arc = SubCenterArcFace(CFG["emb_dim"], n_classes, CFG["subcenters"], CFG["arc_scale"], CFG["arc_margin"]).to(DEVICE)
    return head, arc


head_t, arc_t = fresh_adapters(12345, 8)
opt_t = torch.optim.AdamW(lora_params() + list(head_t.parameters()) + list(arc_t.parameters()), lr=1e-3)
model.train(); head_t.train(); arc_t.train(); times = []
for k in range(3):
    xb = torch.from_numpy(rng0.uniform(-0.1, 0.1, (CFG["batch"], CROP)).astype(np.float32)).to(DEVICE)
    yb = torch.from_numpy(rng0.integers(0, 8, CFG["batch"])).to(DEVICE)
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    loss = arc_t.loss(arc_t.cosine(head_t(tokens(xb, amp=AMP).mean(1))), yb)
    opt_t.zero_grad(set_to_none=True); loss.backward()
    if k == 0:
        assert math.isfinite(float(loss.detach())) and sum(float(w.B.grad.abs().sum()) for w in LORA if w.B.grad is not None) > 0, "stop: no gradient reaches LoRA"
    opt_t.step()
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    times.append(time.time() - t0)
model.eval()
with torch.no_grad():
    moved = float((tokens(x4, amp=False).mean(1) - Z4).abs().max())
    LoRASwitch.on = False
    off = float((tokens(x4, amp=False).mean(1) - Z4).abs().max())
    LoRASwitch.on = True
print(f"after 3 steps: with LoRA on the output moved by {moved:.2e}; with the switch off it differs from frozen BEATs by {off:.2e}")
assert moved > 1e-6 and off < 1e-5, "stop: the LoRA switch does not work"
del head_t, arc_t, opt_t
for w in LORA:
    w.reset()
STEP_S = min(times[1:])
print(f"BEATs iter3+ (AS2M): one training step {STEP_S:.3f} s")

In [ ]:
# 7 · BEATs iter3 (notebook 23's cell, unchanged; notebook 22b's loader and file): load, smoke test, LoRA, GB, a timed step; gate G0(h)
import torchaudio.compliance.kaldi as ta_kaldi


def beats_fb_tokens(m, fb, amp):
    # filter banks (B, frames, 128) of any BEATs model → last-layer tokens (B, T·8, 768): the forward pass of cell 9
    with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=amp):
        f = m.patch_embedding(fb.unsqueeze(1))
        f = f.reshape(f.shape[0], f.shape[1], -1).transpose(1, 2)
        f = m.layer_norm(f)
        if m.post_extract_proj is not None:
            f = m.post_extract_proj(f)
        h, _ = m.encoder(m.dropout_input(f), padding_mask=None)
    return h.float()


def kaldi_frames(n):
    # frames of a 25 ms / 10 ms Kaldi filter bank of n samples (edges snipped)
    return 1 + (n - 400) // 160


class LoRAQKV(nn.Module):
    """A fused query-key-value projection with LoRA on the query and value parts only: y = W x + (α / r) · [B_q A_q x, 0, B_v A_v x].
    B starts at zero, so the model starts exactly as the frozen one; LoRASwitch.on switches every block at once."""
    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        d = base.in_features
        assert base.out_features == 3 * d, (base.in_features, base.out_features)
        self.base, self.r, self.scale = base, r, alpha / r
        self.A = nn.Parameter(torch.empty(2, r, d, device=base.weight.device))
        self.B = nn.Parameter(torch.empty(2, d, r, device=base.weight.device))
        self.drop = nn.Dropout(dropout)
        self.reset()

    def reset(self):
        with torch.no_grad():
            for i in range(2):
                nn.init.kaiming_uniform_(self.A[i], a=math.sqrt(5))
            self.B.zero_()

    @property
    def weight(self):
        return self.base.weight

    @property
    def bias(self):
        return self.base.bias

    def forward(self, x):
        y = self.base(x)
        if not LoRASwitch.on:
            return y
        h = self.drop(x)
        dq = F.linear(F.linear(h, self.A[0]), self.B[0]) * self.scale
        dv = F.linear(F.linear(h, self.A[1]), self.B[1]) * self.scale
        return y + torch.cat([dq, torch.zeros_like(dq), dv], -1)


class Enc:
    # one encoder: waveforms → last-layer patch tokens (no CLS), its LoRA blocks, its patch layout, and pooling over the time
    # patches that start inside the clip
    def __init__(self, key, net, lora, info=None):
        self.key, self.name, self.dim, self.net, self.lora = key, ENCODERS[key]["name"], ENCODERS[key]["dim"], net, list(lora)
        self.layout, self.ok_lora, self.step_s, self.info = "frequency fastest", bool(lora), None, dict(info or {})

    def tokens(self, x, amp):
        with torch.autocast(device_type=DEVICE, enabled=False):
            fb = self.wave2fb(x.float())                              # filter banks always in float32
        return self.fb2tok(fb, amp)

    def grid(self, h, n_samples):
        # (B, L, D) tokens → (B, T, 8, D): the time patches that start inside a clip of n_samples, in this encoder's layout
        B, Lt, D = h.shape; Tp = Lt // 8
        g = h.view(B, Tp, 8, D) if self.layout == "frequency fastest" else h.view(B, 8, Tp, D).transpose(1, 2)
        return g[:, :max(1, min(Tp, -(-kaldi_frames(n_samples) // 16)))]

    def pool(self, h, n_samples):
        return self.grid(h, n_samples).mean((1, 2))

    def train(self):
        self.net.train()

    def eval(self):
        self.net.eval()

    def reset_lora(self):
        for w in self.lora:
            w.reset()


class BeatsEnc(Enc):
    def wave2fb(self, x):
        return self.net.preprocess(x)

    def fb2tok(self, fb, amp):
        return beats_fb_tokens(self.net, fb, amp)


class EatEnc(Enc):
    varlen = False

    def wave2fb(self, x):
        # EAT's model card: waveform minus its mean; Kaldi filter banks (128 mel, Hanning, 10 ms shift, no dither, HTK-compatible);
        # zero-padded to 1,024 frames (training crops: to a multiple of 16 frames when the model accepts it); then normalised
        out = []
        for w in x:
            w = w - w.mean()
            out.append(ta_kaldi.fbank(w.unsqueeze(0), htk_compat=True, sample_frequency=SR, use_energy=False, window_type="hanning",
                                      num_mel_bins=128, dither=0.0, frame_shift=10))
        fb = torch.stack(out); n = fb.shape[1]
        tgt = EAT_FRAMES if (n > EAT_FRAMES - 128 or not self.varlen) else 16 * math.ceil(n / 16)
        fb = F.pad(fb, (0, 0, 0, tgt - n)) if tgt >= n else fb[:, :tgt]
        return (fb - EAT_NORM[0]) / (2.0 * EAT_NORM[1])

    def fb2tok(self, fb, amp):
        with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=amp):
            out = self.net.extract_features(fb.unsqueeze(1))
        if isinstance(out, dict):
            out = out["x"] if "x" in out else next(iter(out.values()))
        if isinstance(out, (tuple, list)):
            out = out[0]
        return out[:, 1:].float()                                     # without the CLS token


class StandInAttn(nn.Module):
    # TEST only: attention with one fused qkv projection, like EAT's
    def __init__(self, dim, heads=2):
        super().__init__()
        self.h, self.qkv, self.proj = heads, nn.Linear(dim, 3 * dim), nn.Linear(dim, dim)

    def forward(self, x):
        B, N, C = x.shape
        q, k, v = self.qkv(x).view(B, N, 3, self.h, C // self.h).permute(2, 0, 3, 1, 4)
        return self.proj(F.scaled_dot_product_attention(q, k, v).transpose(1, 2).reshape(B, N, C))


class StandInBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.n1, self.attn, self.n2 = nn.LayerNorm(dim), StandInAttn(dim), nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, 2 * dim), nn.GELU(), nn.Linear(2 * dim, dim))

    def forward(self, x):
        x = x + self.attn(self.n1(x))
        return x + self.mlp(self.n2(x))


class EatStandIn(nn.Module):
    # TEST only: a small ViT with EAT's interface: extract_features([B, 1, T, 128]) → CLS + (T/16 × 8) patch tokens
    def __init__(self, dim, layers):
        super().__init__()
        self.patch, self.cls = nn.Conv2d(1, dim, 16, 16), nn.Parameter(torch.zeros(1, 1, dim))
        self.pos = nn.Parameter(torch.randn(1, 64 * 8, dim) * 0.02)
        self.blocks, self.norm = nn.ModuleList(StandInBlock(dim) for _ in range(layers)), nn.LayerNorm(dim)

    def extract_features(self, mel):
        x = self.patch(mel).flatten(2).transpose(1, 2)
        x = torch.cat([self.cls.expand(x.shape[0], -1, -1), x + self.pos[:, :x.shape[1]]], 1)
        for b in self.blocks:
            x = b(x)
        return self.norm(x)


def onedrive_direct(share_url):
    # OneDrive share link → direct download (the documented 'shares' form of the OneDrive API), as notebook 1
    b64 = base64.b64encode(share_url.encode()).decode().rstrip("=").replace("/", "_").replace("+", "-")
    return f"https://api.onedrive.com/v1.0/shares/u!{b64}/root/content"


def attach_beats_lora(m):
    lora = []
    for layer in m.encoder.layers:
        att = layer.self_attn
        for nm in ("q_proj", "v_proj"):
            w = LoRASwitch(getattr(att, nm), CFG["lora_rank"], CFG["lora_alpha"], CFG["lora_dropout"]); setattr(att, nm, w); lora.append(w)
    return lora


WN = ((".parametrizations.weight.original0", ".weight_g"), (".parametrizations.weight.original1", ".weight_v"))


def tensors_of(ck):
    # the weights of a loaded checkpoint (a bare state dict, or under "model" or "state_dict") and its configuration, if it has one
    if isinstance(ck, dict):
        for k in ("model", "state_dict"):
            if isinstance(ck.get(k), dict):
                return ck[k], ck.get("cfg")
        if ck and all(torch.is_tensor(v) for v in ck.values()):
            return ck, None
    raise RuntimeError(f"no weights in the file (it holds a {type(ck).__name__}" + (f" with keys {list(ck)[:5]})" if isinstance(ck, dict) else ")"))


def wn_names(sd, own_keys):
    # torch names weight norm's two tensors in two ways (weight_g / weight_v, and parametrizations); the file's keys take the model's
    old = any(k.endswith(".weight_g") for k in own_keys)
    out = {}
    for k, v in sd.items():
        for a, b in WN:
            k = k.replace(a, b) if old else k.replace(b, a)
        out[k] = v
    return out


def match_weights(net, sd):
    # Amendment 1: the file's keys on the model's own, as they are or with one leading prefix removed or added (whichever matches
    # the most keys). Every trainable parameter must come from the file; fixed tensors (buffers, and parameters without gradient)
    # may come from the model's own construction; keys of the file that the model does not have are not used. Both are listed.
    own = net.state_dict(); keys = set(own)
    trainable = {n for n, q in net.named_parameters() if q.requires_grad}
    sd = wn_names(sd, keys)
    cands = {("as is", "")}
    for k in sd:
        for o in keys:
            if k.endswith("." + o):
                cands.add(("removed", k[:len(k) - len(o)]))
            elif o.endswith("." + k):
                cands.add(("added", o[:len(o) - len(k)]))

    def mapping(how, p):
        return {x[len(p):]: x for x in sd if x.startswith(p)} if how == "removed" else {p + x: x for x in sd}

    how, p = max(sorted(cands), key=lambda c: (len(set(mapping(*c)) & keys), c[0] == "as is"))
    m = {k: x for k, x in mapping(how, p).items() if k in keys}
    other = [k for k in m if tuple(sd[m[k]].shape) != tuple(own[k].shape)]
    missing = sorted(keys - set(m)); lost = [k for k in missing if k in trainable]
    unused = sorted(set(sd) - set(m.values()))
    if other or lost:
        raise RuntimeError(f"the file's weights do not fit the model: {len(lost)} trainable parameters are not in the file (e.g. {lost[:3]}), "
                           f"{len(other)} have another shape (e.g. {other[:3]})")
    rep = {"keys_loaded": len(m), "key_prefix": f"'{p}' {how}" if p else "none", "fixed_tensors_from_construction": missing,
           "file_keys_not_used": len(unused), "file_keys_not_used_e_g": unused[:5]}
    return {k: sd[x] for k, x in m.items()}, rep


def load_matched(net, sd):
    state, rep = match_weights(net, sd)
    res = net.load_state_dict(state, strict=False)
    assert not res.unexpected_keys and sorted(res.missing_keys) == rep["fixed_tensors_from_construction"], res
    return rep


def beats_iter3_file():
    # a BEATs iter3 file already in Drive is used as it is; otherwise the authors' link, the Azure copy, then PretrainedSED's copy
    spec = ENCODERS["BEATs-iter3"]; d = WORK if TEST else CKPT_DIR
    own, sed = d / spec["file"], d / spec["sed_file"]
    for f in (own, sed):
        if f.exists() and f.stat().st_size < 10_000_000:
            f.unlink()                                                # an error page saved by a failed attempt, not the model
    if not (own.exists() or sed.exists()):
        for label, url, dest in (("the authors' link (OneDrive)", spec["share"] and onedrive_direct(spec["share"]), own),
                                 ("Microsoft's Azure copy", spec["azure"], own), ("PretrainedSED's release v0.0.1", spec["sed"], sed)):
            if not url:
                continue
            try:
                download(url, dest)
            except Exception as e:
                print(f"BEATs iter3 · {label}: failed ({e.__class__.__name__})")
            if dest.exists() and dest.stat().st_size < 10_000_000:
                dest.unlink()
            if dest.exists():
                print(f"BEATs iter3 · {label}: downloaded", flush=True)
                break
    for f, label in ((own, "the authors' file"), (sed, "PretrainedSED's copy (release v0.0.1); provenance caveat in Amendment 1")):
        if f.exists():
            return f, label
    raise RuntimeError("no download of BEATs iter3 worked (the authors' link, the Azure copy, PretrainedSED's copy)")


def load_beats_iter3():
    f, label = beats_iter3_file()
    ck = torch.load(f, map_location="cpu", weights_only=True)          # tensors and plain values only: no code in the file runs
    sd, cfg_f = tensors_of(ck)
    cfg_ = cfg_f if cfg_f is not None else {**chk["cfg"], "finetuned_model": False}   # weights only: our iter3+ architecture, no classifier
    m = BEATs(BEATsConfig(cfg_))
    rep = load_matched(m, sd)
    msd, ref = m.state_dict(), chk["model"]
    com = [k for k in msd if k in ref and tuple(ref[k].shape) == tuple(msd[k].shape)]
    dist = math.sqrt(sum(float((msd[k].double() - ref[k].double()).pow(2).sum()) for k in com)
                     / max(sum(float(ref[k].double().pow(2).sum()) for k in com), 1e-30))
    m.eval().to(DEVICE)
    for q in m.parameters():
        q.requires_grad_(False)
    info = {"checkpoint_sha256": sha256(f), "file": f.name, "source": label,
            "configuration": "the file's own" if cfg_f is not None else "our iter3+ checkpoint's, classifier off (the file holds weights only)",
            "fine_tuned_on_audioset": bool(getattr(BEATsConfig(cfg_), "finetuned_model", False)),
            "relative_distance_to_our_iter3plus": round(dist, 4), "tensors_compared": len(com), **rep}
    print(f"BEATs iter3 · {label} · SHA-256 {info['checkpoint_sha256'][:16]}… · relative distance from our iter3+ weights {dist:.4f}")
    return BeatsEnc("BEATs-iter3", m, attach_beats_lora(m), info)


def eat_snapshot(repo):
    # the repository's files at one commit: configuration, code and weights (TEST: a local folder)
    if os.path.isdir(repo):
        return repo, "a local folder (TEST)"
    from huggingface_hub import HfApi, snapshot_download
    names = HfApi().list_repo_files(repo)
    pats = ["*.json", "*.py", "*.txt"] + (["*.safetensors"] if any(n.endswith(".safetensors") for n in names) else ["*.bin"])
    path = snapshot_download(repo, allow_patterns=pats)
    return path, Path(path).name                                      # the snapshot folder is named by its commit


def read_weights(path):
    # every weights file of the snapshot: safetensors, else pytorch_model*.bin read with weights_only=True
    files, sd = sorted(Path(path).glob("*.safetensors")), {}
    if files:
        from safetensors.torch import load_file
        for f in files:
            sd.update(load_file(str(f)))
    else:
        files = sorted(Path(path).glob("pytorch_model*.bin"))
        for f in files:
            sd.update(torch.load(f, map_location="cpu", weights_only=True))
    if not sd:
        raise RuntimeError("no weights file (safetensors or pytorch_model.bin) in the repository")
    return sd, [f.name for f in files]


def load_remote_model(path):
    # from_pretrained first; if it fails (transformers 5 with remote code that does not call post_init), the model is built from
    # the repository's own configuration and code, and its weights file is loaded into it (Amendment 1). A package that its
    # code needs is installed, then the load is tried again.
    from transformers import AutoConfig, AutoModel
    from transformers.dynamic_module_utils import get_class_from_dynamic_module
    for _ in range(4):
        try:
            try:
                return AutoModel.from_pretrained(path, trust_remote_code=True), {"loaded_by": "from_pretrained"}
            except ImportError:
                raise
            except Exception as e:
                first = repr(e)[:300]
            print(f"  from_pretrained failed ({first[:140]}); building the model from its own configuration and code", flush=True)
            conf = AutoConfig.from_pretrained(path, trust_remote_code=True)
            net = get_class_from_dynamic_module(conf.auto_map["AutoModel"], path)(conf)
            sd, files = read_weights(path)
            rep = load_matched(net, sd)
            return net, {"loaded_by": "built from its own configuration and code; weights loaded directly", "from_pretrained_error": first,
                         "weights_files": files, **rep}
        except ImportError as e:
            if isinstance(e, ModuleNotFoundError) and e.name:
                need = [e.name.split(".")[0]]
            else:
                need = " ".join(re.findall(r"pip install ([^`\n]+)", str(e))[:1]).split()
            if not need:
                raise
            print(f"  its code needs {need}; installing", flush=True)
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)
            importlib.invalidate_caches()
    raise RuntimeError("did not load")


def load_eat(key):
    spec = ENCODERS[key]
    if TEST and not os.path.isdir(str(spec.get("repo"))):
        torch.manual_seed(ENC_ORDER.index(key)); net, info = EatStandIn(spec["dim"], spec["layers"]), {"commit": "TEST stand-in"}
    else:
        try:
            import transformers
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers"], check=True)
            import transformers
        path, commit = eat_snapshot(spec["repo"])
        net, info = load_remote_model(path)
        info.update(commit=commit, transformers=transformers.__version__)
    net = net.float().to(DEVICE).eval()
    for q in net.parameters():
        q.requires_grad_(False)
    info = {"repo": spec.get("repo"), **info, "parameters_millions": round(sum(q.numel() for q in net.parameters()) / 1e6, 1)}
    fused = [(n, m) for n, m in net.named_modules() if isinstance(m, nn.Linear) and n.split(".")[-1] == "qkv"]
    split = [(n, m) for n, m in net.named_modules() if isinstance(m, nn.Linear) and n.split(".")[-1] in ("q_proj", "v_proj", "query", "value")]
    lora = []
    if len(fused) == spec["layers"]:
        for n, m in fused:
            parent = net.get_submodule(n.rsplit(".", 1)[0]) if "." in n else net
            w = LoRAQKV(m, CFG["lora_rank"], CFG["lora_alpha"], CFG["lora_dropout"]); setattr(parent, n.split(".")[-1], w); lora.append(w)
        info["lora_on"] = f"{len(fused)} fused query-key-value projections (query and value parts)"
    elif len(split) == 2 * spec["layers"]:
        for n, m in split:
            parent = net.get_submodule(n.rsplit(".", 1)[0]) if "." in n else net
            w = LoRASwitch(m, CFG["lora_rank"], CFG["lora_alpha"], CFG["lora_dropout"]); setattr(parent, n.split(".")[-1], w); lora.append(w)
        info["lora_on"] = f"{len(split)} separate query and value projections"
    else:
        info["lora_problem"] = f"found {len(fused)} fused qkv and {len(split)} separate query/value projections; {spec['layers']} layers expected"
    return EatEnc(key, net, lora, info)


def lora_params_of(enc):
    return [p for w in enc.lora for p in (w.A, w.B)]


def smoke(enc, timed=True):
    # tokens: shape, finiteness; EAT: other lengths; LoRA: unchanged at zero, a gradient reaches it, the switch works; a timed step
    rep = {}
    x10 = torch.from_numpy(rng0.uniform(-0.1, 0.1, (2, 10 * SR)).astype(np.float32)).to(DEVICE)
    with torch.no_grad():
        h = enc.tokens(x10, amp=False)
    tp = EAT_FRAMES // 16 if isinstance(enc, EatEnc) else kaldi_frames(10 * SR) // 16
    rep["tokens_of_10_s"] = list(h.shape)
    if not (h.shape[1] == tp * 8 and h.shape[2] == enc.dim and bool(torch.isfinite(h).all())):
        raise RuntimeError(f"{enc.key}: tokens {tuple(h.shape)}, expected (2, {tp * 8}, {enc.dim}) and finite")
    if isinstance(enc, EatEnc):
        enc.varlen = True
        try:
            with torch.no_grad():
                hc = enc.tokens(x10[:, :CROP], amp=False)
            enc.varlen = bool(hc.shape[1] == 16 * math.ceil(kaldi_frames(CROP) / 16) // 16 * 8 and torch.isfinite(hc).all())
        except Exception as e:
            enc.varlen = False; rep["other_length_error"] = repr(e)[:200]
        rep["accepts_other_lengths"] = enc.varlen
    if not enc.lora:
        enc.ok_lora = False; rep["lora"] = enc.info.get("lora_problem", "no LoRA"); return rep
    try:
        xs = x10[:, :CROP]
        with torch.no_grad():
            LoRASwitch.on = False; z0 = enc.pool(enc.tokens(xs, amp=False), CROP)
            LoRASwitch.on = True; z1 = enc.pool(enc.tokens(xs, amp=False), CROP)
        scale = float(z0.abs().mean()) + 1e-12
        assert float((z1 - z0).abs().max()) / scale < 1e-4, "LoRA at zero changed the output"
        head_t = Head(enc.dim, CFG["emb_dim"]).to(DEVICE)
        arc_t = SubCenterArcFace(CFG["emb_dim"], 8, CFG["subcenters"], CFG["arc_scale"], CFG["arc_margin"]).to(DEVICE)
        opt_t = torch.optim.AdamW(lora_params_of(enc) + list(head_t.parameters()) + list(arc_t.parameters()), lr=1e-3)
        enc.train(); head_t.train(); arc_t.train(); times = []
        for k in range(3):
            xb = torch.from_numpy(rng0.uniform(-0.1, 0.1, (CFG["batch"], CROP)).astype(np.float32)).to(DEVICE)
            yb = torch.from_numpy(rng0.integers(0, 8, CFG["batch"])).to(DEVICE)
            if DEVICE == "cuda":
                torch.cuda.synchronize()
            t0 = time.time()
            loss = arc_t.loss(arc_t.cosine(head_t(enc.pool(enc.tokens(xb, amp=AMP), CROP))), yb)
            opt_t.zero_grad(set_to_none=True); loss.backward()
            if k == 0:
                assert math.isfinite(float(loss.detach())) and sum(float(w.B.grad.abs().sum()) for w in enc.lora if w.B.grad is not None) > 0, "no gradient reaches LoRA"
            opt_t.step()
            if DEVICE == "cuda":
                torch.cuda.synchronize()
            times.append(time.time() - t0)
        enc.eval()
        with torch.no_grad():
            moved = float((enc.pool(enc.tokens(xs, amp=False), CROP) - z0).abs().max()) / scale
            LoRASwitch.on = False
            off = float((enc.pool(enc.tokens(xs, amp=False), CROP) - z0).abs().max()) / scale
            LoRASwitch.on = True
        assert moved > 1e-6 and off < 1e-4, f"the LoRA switch does not work (moved {moved:.1e}, off {off:.1e})"
        enc.step_s = min(times[1:]) if timed else None
        rep["lora"] = enc.info.get("lora_on", "LoRA"); rep["step_seconds"] = round(enc.step_s, 4) if enc.step_s else None
        del head_t, arc_t, opt_t
    except Exception as e:
        enc.ok_lora = False; rep["lora"] = f"failed: {repr(e)[:300]}"
    finally:
        LoRASwitch.on = True; enc.reset_lora(); enc.eval()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    return rep


def gate_gb(enc):
    # GB: extra energy in the top (bottom) 16 mel bins over the middle third, against the same input without it; the change of
    # each token, averaged per frequency row under each candidate layout
    with torch.no_grad():
        base = enc.wave2fb(torch.from_numpy(np.random.default_rng(5).normal(0, 0.003, (1, 10 * SR)).astype(np.float32)).to(DEVICE))
        a, b = base.shape[1] // 3, 2 * base.shape[1] // 3
        top, bot = base.clone(), base.clone()
        top[:, a:b, 112:128] += 2.0; bot[:, a:b, 0:16] += 2.0
        LoRASwitch.on = False
        try:
            h0, ht, hb = (enc.fb2tok(z, amp=False) for z in (base, top, bot))
        finally:
            LoRASwitch.on = True
    Lt = h0.shape[1]; Tp = Lt // 8
    dt = (ht - h0).norm(dim=-1)[0].cpu().numpy(); db = (hb - h0).norm(dim=-1)[0].cpu().numpy()
    out = {}
    for lay, row in (("frequency fastest", np.arange(Lt) % 8), ("time fastest", np.arange(Lt) // Tp)):
        pt = np.array([dt[row == f].mean() for f in range(8)]); pb = np.array([db[row == f].mean() for f in range(8)])
        it, ib = int(pt.argmax()), int(pb.argmax())
        rt = float(pt[it] / max(np.delete(pt, it).mean(), 1e-12)); rb = float(pb[ib] / max(np.delete(pb, ib).mean(), 1e-12))
        out[lay] = {"top_row": it, "bottom_row": ib, "ratio_top": round(rt, 2), "ratio_bottom": round(rb, 2),
                    "confirmed": bool({it, ib} == {0, 7} and min(rt, rb) >= GB_RATIO)}   # the two ends of the spectrum, one each
    if out["time fastest"]["confirmed"] and not out["frequency fastest"]["confirmed"]:
        enc.layout = "time fastest"
    out["used"] = enc.layout
    return out


ENC, FAILED, GBR = {}, {}, {}
ENC["BEATs"] = BeatsEnc("BEATs", model, LORA, {"checkpoint_sha256": CKPT_SHA})        # cell 9's model and LoRA blocks
ENC["BEATs"].step_s = STEP_S
with torch.no_grad():
    _gap = float((ENC["BEATs"].tokens(x4, amp=False) - tokens(x4, amp=False)).abs().max())
    _pool = float((ENC["BEATs"].pool(tokens(x4, amp=False), x4.shape[1]) - tokens(x4, amp=False).mean(1)).abs().max())
print(f"BEATs through the encoder wrapper against cell 9's forward pass: {_gap:.1e} (pooling {_pool:.1e})")
assert _gap < 1e-6 and _pool < 1e-5, "stop: the encoder wrapper does not reproduce cell 9"
for key in NEW_ENCODERS:
    t0 = time.time()
    try:
        enc = load_beats_iter3() if key == "BEATs-iter3" else load_eat(key)
        enc.info.update(smoke(enc))
        ENC[key] = enc
        print(f"{key}: ready · {json.dumps(enc.info)} · {time.time() - t0:.0f} s", flush=True)
    except Exception as e:
        FAILED[key] = repr(e)[:500]
        print(f"{key}: DROPPED, our loader failed ({FAILED[key]})", flush=True)
    gc.collect()
for key, enc in ENC.items():
    GBR[key] = gate_gb(enc)
    print(f"GB · {key}: {json.dumps(GBR[key])}")


# G0(h): the BEATs iter3 file is the one notebook 22b used
G0H = {"sha256": ENC["BEATs-iter3"].info.get("checkpoint_sha256") if "BEATs-iter3" in ENC else None, "expected": B3_SHA_N22B}
print("G0(h) · the BEATs iter3 file is notebook 22b's:", G0H["sha256"] == B3_SHA_N22B)
if not TEST and G0H["sha256"] != B3_SHA_N22B:
    raise SystemExit("stop: G0(h) failed, the BEATs iter3 file is not the one notebook 22b used (or it did not load)")

In [ ]:
# 8 · ECHO, frozen (notebook 23's cell, unchanged; notebook 20's cell: notebook 15's loader, as on the model card); gate G0(g), its weights
INFO = {}
def load_echo():
    # as on the ECHO model card: its own code and model.safetensors from the same Hugging Face snapshot
    from huggingface_hub import snapshot_download
    from safetensors.torch import load_file
    snap = Path(snapshot_download(ECHO_HF))
    INFO["ECHO"] = {"repo": ECHO_HF, "revision": snap.name, "weights_sha256": sha256(snap / "model.safetensors"),
                    "code_sha256": sha256(snap / "audioMAE_band_upgrade.py")}
    saved = {k: sys.modules.pop(k) for k in ("modules", "utils") if k in sys.modules}
    sys.path.insert(0, str(snap))
    try:
        mod = importlib.import_module("audioMAE_band_upgrade")
    finally:
        sys.path.remove(str(snap))
        for k in ("modules", "utils"):
            sys.modules.pop(k, None)
        sys.modules.update(saved)
    m = mod.AudioMAEWithBand(spec_len=2000, band_width=32, shift_size=16, in_chans=1, embed_dim=384, encoder_depth=12,
                             num_heads=6, mlp_ratio=4.0, freq_pos_emb_dim=384)
    missing, unexpected = m.load_state_dict(load_file(str(snap / "model.safetensors")), strict=False)
    assert not missing, f"stop: ECHO weights missing for {list(missing)[:5]}"
    INFO["ECHO"].update(loader="model card: its own code + model.safetensors", unexpected_keys=len(unexpected))
    return m.to(DEVICE).eval()




@torch.no_grad()
def emb_echo(m, waves):
    out = []
    for x in waves:
        u, _ = m.extract_features_from_audio(torch.from_numpy(np.ascontiguousarray(x, dtype=np.float32))[None].to(DEVICE), sample_rate=SR)
        out.append(u.float().cpu().numpy())
    return np.stack(out)


class EchoStandIn(torch.nn.Module):
    # TEST ONLY: a fixed random projection of log band energies, shaped like ECHO's output. Never used on Colab.
    def __init__(self, seed=2, bands=7, dim=16):
        super().__init__()
        g = torch.Generator().manual_seed(seed)
        self.bands, self.P = bands, torch.randn(bands, 8, dim, generator=g)

    def extract_features_from_audio(self, x, sample_rate):
        S = torch.stft(x[0].float(), n_fft=400, hop_length=160, window=torch.hann_window(400, device=x.device), return_complex=True).abs().log1p().T
        e = torch.stack([b.mean(-1) for b in S.chunk(self.bands, dim=-1)], -1)
        q = torch.stack([e[i * e.shape[0] // 8:(i + 1) * e.shape[0] // 8].mean(0) for i in range(8)], 0)
        return torch.einsum("qk,kqd->kd", q, self.P.to(x.device)).flatten(), None


if TEST:
    ECHO = EchoStandIn().to(DEVICE).eval(); INFO["ECHO"] = {"repo": "stand-in (TEST)", "weights_sha256": "", "loader": "stand-in"}
else:
    ECHO = load_echo()
    if INFO["ECHO"]["weights_sha256"] != ECHO_SHA_N15:
        print("NOTE: the ECHO weights differ from notebook 15's (gate G0(g))")

G0G = {"weights_sha256": INFO["ECHO"].get("weights_sha256"), "expected": ECHO_SHA_N15, "revision": INFO["ECHO"].get("revision")}
print("ECHO:", json.dumps(INFO["ECHO"]))
if not TEST and G0G["weights_sha256"] != ECHO_SHA_N15:
    raise SystemExit("stop: G0(g) failed, the ECHO weights are not notebook 15's")


@torch.no_grad()
def echo_stream(mm, rows=None, chunk=64):
    # ECHO on whole clips of a stream, a few at a time → (n, 7 × 384) float32
    rows = np.arange(mm.shape[0]) if rows is None else rows
    return np.concatenate([emb_echo(ECHO, [f32(mm[r]) for r in rows[i:i + chunk]]) for i in range(0, len(rows), chunk)])

# gate G0(c) of this notebook's rules: the BEATs code files, both BEATs checkpoints and ECHO's weights are notebook 23's (SHA-256).
# The two checkpoints and ECHO were already checked by the cells above; the code files are checked here
G0C = {"code_files": {f: bool(code_sha[f] == h) for f, h in CODE_SHA256.items()}, "beats_iter3plus": bool(CKPT_SHA == CKPT_SHA_N1),
       "beats_iter3": bool(G0H["sha256"] == B3_SHA_N22B), "echo_weights": bool(G0G["weights_sha256"] == ECHO_SHA_N15)}
G0C["ok"] = bool(all(G0C["code_files"].values()) and G0C["beats_iter3plus"] and G0C["beats_iter3"] and G0C["echo_weights"])
print("G0(c) · the BEATs code files, both BEATs checkpoints and ECHO's weights are notebook 23's:", G0C["ok"], G0C)
if not TEST and not G0C["ok"]:
    raise SystemExit("stop: G0(c) failed, a code file or a model file is not the one notebook 23 used")

In [ ]:
# 9 · DCASE 2025's own files of the fifteen types (MD5 against the Zenodo records), and every clip's first 10 s in one int16 store per
# type on the local disk; gates G0(a) and G0(b). The readers are notebook 23's (through notebook 25). The loader is notebook 28's: a
# development type has one zip, an evaluation type two; the supplemental clips are not read.
CHECKED = {}                                        # files whose MD5 was already checked in this session


def get_file(entry):
    url, name, want = entry
    dest = WORK / name
    if CHECKED.get(name) == want and dest.exists():
        return dest
    if dest.exists() and md5(dest) == want:
        CHECKED[name] = want
        return dest
    t = time.time(); download(url, dest)
    got = md5(dest)
    print(f"{name}: downloaded in {time.time() - t:.0f} s · MD5 matches Zenodo: {got == want}", flush=True)
    if got != want:
        dest.unlink()
        raise SystemExit(f"stop: {name} is not the published file")
    CHECKED[name] = want
    return dest


WAV_FORMATS = Counter()                                               # the sample formats of the WAV files read in this session


def read_all(data):
    # WAV bytes (plain or WAVE_FORMAT_EXTENSIBLE; PCM of 16, 24 or 32 bits, or 32-bit float) -> (samples, channels) float32 in [-1, 1], sample rate
    assert data[:4] == b"RIFF" and data[8:12] == b"WAVE", "not a WAV file"
    pos, fmt, raw = 12, None, None
    while pos + 8 <= len(data):
        cid, size = data[pos:pos + 4], struct.unpack("<I", data[pos + 4:pos + 8])[0]
        body = data[pos + 8:pos + 8 + size]
        if cid == b"fmt ":
            tag, ch, sr, _, _, bits = struct.unpack("<HHIIHH", body[:16])
            if tag == 0xFFFE:
                tag = struct.unpack("<H", body[24:26])[0]
            fmt = (tag, ch, sr, bits)
        elif cid == b"data":
            raw = body
        pos += 8 + size + (size & 1)
    tag, ch, sr, bits = fmt
    assert (tag, bits) in ((1, 16), (1, 24), (1, 32), (3, 32)), f"expected PCM of 16, 24 or 32 bits or 32-bit float, got format {tag}, {bits}-bit"
    WAV_FORMATS[f"{'float' if tag == 3 else 'PCM'} {bits}-bit"] += 1
    w = bits // 8; nf = len(raw) // (w * ch); raw = raw[:nf * w * ch]
    if (tag, bits) == (1, 16):
        return np.frombuffer(raw, dtype="<i2").reshape(-1, ch).astype(np.float32) / 32768.0, sr
    if (tag, bits) == (1, 24):                                        # three bytes per sample, little-endian, signed: a 24-bit value is exact in float32
        b = np.frombuffer(raw, dtype=np.uint8).reshape(-1, 3).astype(np.int32)
        v = b[:, 0] | (b[:, 1] << 8) | (b[:, 2] << 16); v = np.where(v >= 1 << 23, v - (1 << 24), v)
        return (v.astype(np.float32) / 8388608.0).reshape(-1, ch), sr
    if (tag, bits) == (1, 32):
        return (np.frombuffer(raw, dtype="<i4").astype(np.float64) / 2147483648.0).astype(np.float32).reshape(-1, ch), sr
    return np.frombuffer(raw, dtype="<f4").reshape(-1, ch).astype(np.float32), sr


def stream(fn, items, depth=8, workers=2):
    # fn(item) runs in background threads, results come back in order: unzipping overlaps the main thread's work
    with ThreadPoolExecutor(workers) as ex:
        q, it = deque(), iter(items)
        for item in it:
            q.append(ex.submit(fn, item))
            if len(q) >= depth:
                break
        while q:
            res = q.popleft().result()
            for item in it:
                q.append(ex.submit(fn, item))
                break
            yield res


def to_int16(v):
    # exact for a 16-bit WAV file; a clip of more bits is rounded to 16 (to the nearest, ties to even) and clipped to the int16 range
    return np.clip(np.round(v * 32768.0), -32768, 32767).astype(np.int16)


def f32(a):
    # a slice of a stream → float32 signal: int16 streams hold samples × 32768, float16 streams hold the signal itself
    a = np.asarray(a)
    return a.astype(np.float32) / 32768.0 if a.dtype == np.int16 else a.astype(np.float32)


STORE = WORK / "stores30"; STORE.mkdir(exist_ok=True)
DST, DMETA = {}, {}


def dcase_load(t):
    # every DCASE 2025 clip of type t in one int16 store of NS_RULE samples (10 s): the first 10 s of a longer clip, as the view's reference
    # code cuts it, or a shorter clip repeated from its start; a development type's train and test clips, or an evaluation type's training clips and its test clips. The supplemental clips are not read
    meta_f, arr_f = STORE / f"dc_{t}.json", STORE / f"dc_{t}.npy"
    if meta_f.exists() and arr_f.exists():
        DMETA[t] = json.loads(meta_f.read_text()); DST[t] = np.load(arr_f, mmap_mode="r")
        return
    zfs = {k_: zipfile.ZipFile(get_file(z_[t])) for k_, z_ in (("dev", DC_DEV), ("evaltrain", XZIPS), ("evaltest", DC_EVAL)) if t in z_}
    rows, skipped = [], []
    for kind, zf in zfs.items():
        for nm in sorted(x for x in zf.namelist() if x.lower().endswith(".wav")):
            parts = nm.replace("\\", "/").split("/"); base, folder = parts[-1], parts[-2]
            if "__MACOSX" in parts or base.startswith("._"):          # an archive's metadata entry, not a clip (none is expected)
                skipped.append(nm); continue
            if folder not in ("train", "test"):                       # the supplemental clips (clean machine sound or noise only) are not used
                continue
            part = folder
            # the name of a training clip and of a development test clip carries its domain and its kind; an evaluation test clip's carries neither
            said = re.match(r"section_\d+_(source|target)_(train|test)_(normal|anomaly)_", base)
            label = (1 if (said and said.group(3) == "anomaly") else 0) if (kind == "dev" and part == "test") else (0 if part == "train" else -1)
            dom = said.group(1) if said else ""
            rows.append((kind, nm, "dev" if kind == "dev" else "eval", part, re.search(r"(section_\d+)", base).group(1), label, base, dom))
    ids = {(s_, p_): sorted(set(r[4] for r in rows if r[2] == s_ and r[3] == p_)) for s_ in ("dev", "eval") for p_ in ("train", "test")}
    if not (ids[("dev", "train")] == ids[("dev", "test")] == DEV_IDS[t] and ids[("eval", "train")] == ids[("eval", "test")] == EVAL_IDS[t]):
        raise SystemExit(f"stop: G0(b) failed. The sections of {t} are not the rules': {ids}")
    assert not any(r[2] == "eval" and r[3] == "train" and "_train_normal_" not in r[6] for r in rows), "the additional training clips are normal"
    ns, lengths, _f0 = NS_RULE, Counter(), Counter(WAV_FORMATS)
    tmp = arr_f.with_name(arr_f.stem + ".part.npy")
    mm = np.lib.format.open_memmap(tmp, mode="w+", dtype=np.int16, shape=(len(rows), ns))
    for i, (x, sr) in enumerate(stream(lambda i: read_all(zfs[rows[i][0]].read(rows[i][1])), range(len(rows)))):
        if not (sr == SR and x.shape[1] == 1 and x.shape[0] >= NS_MIN):
            raise SystemExit(f"stop: G0(b) failed. {rows[i][1]} is not a 16 kHz, one-channel clip of at least {NS_MIN / SR:g} s "
                             f"(rate {sr}, channels {x.shape[1]}, samples {x.shape[0]})")
        lengths[(f"{rows[i][2]}|{rows[i][3]}", int(x.shape[0]))] += 1
        w_ = to_int16(x[:ns, 0])
        mm[i] = w_ if len(w_) == ns else np.tile(w_, -(-ns // len(w_)))[:ns]   # a shorter clip is repeated from its start until it fills the store
    mm.flush(); del mm
    os.replace(tmp, arr_f)
    for zf in zfs.values():
        zf.close()
    DMETA[t] = {"path": [r[1] for r in rows], "name": [r[6] for r in rows], "set": [r[2] for r in rows], "part": [r[3] for r in rows],
                "mid": [r[4] for r in rows], "label": [r[5] for r in rows], "domain": [r[7] for r in rows], "samples": int(ns), "skipped_archive_entries": skipped,
                "formats": {k: int(v - _f0[k]) for k, v in sorted(WAV_FORMATS.items()) if v - _f0[k]},
                "lengths": {k: {str(n_): int(c) for (k2, n_), c in sorted(lengths.items()) if k2 == k} for k in sorted(set(k_ for k_, _ in lengths))},
                "zips": {k_: list(z_[t][1:]) for k_, z_ in (("dev", DC_DEV), ("additional", XZIPS), ("evaluation", DC_EVAL)) if t in z_}}
    meta_f.write_text(json.dumps(DMETA[t]))
    DST[t] = np.load(arr_f, mmap_mode="r")
    if IN_COLAB:
        for z in [z_[t] for z_ in (DC_DEV, XZIPS, DC_EVAL) if t in z_]:
            (WORK / z[1]).unlink(missing_ok=True)


def dc_rows(t, set_, part, q=None):
    m = DMETA[t]
    return np.array([i for i in range(len(m["name"])) if m["set"][i] == set_ and m["part"][i] == part and (q is None or m["mid"][i] == q)], dtype=np.int64)


t_data = time.time()
DCASE = {"counts": {}, "lengths": {}, "domains": {}}
for t in TYPES:
    dcase_load(t)
    _m = DMETA[t]; _mid, _dom = np.array(_m["mid"]), np.array(_m["domain"])
    DCASE["counts"][t] = {f"{s_}|{p_}": {q: int(len(dc_rows(t, s_, p_, q))) for q in sorted(set(_mid[dc_rows(t, s_, p_)].tolist()))}
                          for s_ in ("dev", "eval") for p_ in ("train", "test")}
    DCASE["domains"][t] = {f"{s_}|{p_}": {q: {d_: int((_dom[dc_rows(t, s_, p_, q)] == d_).sum()) for d_ in ("source", "target")}
                                          for q in sorted(set(_mid[dc_rows(t, s_, p_)].tolist()))} for s_, p_ in (("dev", "train"), ("dev", "test"), ("eval", "train"))}
    DCASE["lengths"][t] = _m["lengths"]; DCASE.setdefault("formats", {})[t] = _m.get("formats", "not recorded (a store of an earlier session)")
    print(f"DCASE 2025 · {t}: {len(_m['name'])} clips · training clips per section (source, target): "
          f"{({q: (v['source'], v['target']) for k_ in ('dev|train', 'eval|train') for q, v in DCASE['domains'][t][k_].items()})} · test clips per section: "
          f"{({**DCASE['counts'][t]['dev|test'], **DCASE['counts'][t]['eval|test']})} · samples per clip before the cut (set|part: samples: clips): {DCASE['lengths'][t]}", flush=True)
NS = int(DMETA[TYPES[0]]["samples"])
assert all(int(DMETA[t]["samples"]) == NS for t in TYPES) and NS == NS_RULE, "stop: a store on the local disk was not cut to the rules' length"

# G0(a) · every zip is the published file: get_file stops on another MD5, and a store keeps the names and MD5s of the zips it was built from
_want = {t: {k_: list(z_[t][1:]) for k_, z_ in (("dev", DC_DEV), ("additional", XZIPS), ("evaluation", DC_EVAL)) if t in z_} for t in TYPES}
G0 = {"a": {"ok": bool(all(DMETA[t]["zips"] == _want[t] for t in TYPES)), "zips": _want}}
print(f"G0(a) · the {sum(len(v) for v in _want.values())} zips are the published files (MD5):", G0["a"]["ok"])
if not G0["a"]["ok"]:
    raise SystemExit(f"stop: G0(a) failed. A store in {STORE} was built from other files; delete that folder and run again")


def g0b(t):
    # the files are what the rules assume (16 kHz, one channel and 10 s were checked per file by the loader)
    m = DMETA[t]; nm, st, pt, lb, mid, dom = (np.array(m[k]) for k in ("name", "set", "part", "label", "mid", "domain"))
    c = DCASE["counts"][t]; tr, ev_test = pt == "train", (st == "eval") & (pt == "test")
    assert (DEV_IDS[t] == [SECTION]) != (EVAL_IDS[t] == [SECTION])                # a type is a development type or an evaluation type
    first_target = {}                                                 # per training section: how many source-domain clips come before its first target-domain clip by name
    for s_, ids in (("dev", DEV_IDS[t]), ("eval", EVAL_IDS[t])):
        for q in ids:
            rows = dc_rows(t, s_, "train", q); d_ = dom[rows[np.argsort(nm[rows], kind="stable")]]
            first_target[q] = int(np.argmax(d_ == "target")) if (d_ == "target").any() else -1
    both = all(int(((st == "dev") & (pt == "test") & (mid == q) & (lb == k_) & (dom == d_)).sum()) > 0 for q in DEV_IDS[t] for k_ in (0, 1) for d_ in ("source", "target"))
    _num = sorted(int(n_[-8:-4]) for n_ in nm[ev_test] if re.fullmatch(r"section_\d\d_\d{4}\.wav", n_))   # 2025's names: section_00_NNNN.wav, from 0000 or from 0001
    numbered = (not EVAL_IDS[t]) or (len(_num) == int(ev_test.sum()) > 0 and _num == list(range(_num[0], _num[0] + len(_num))))
    out = {"sections": bool(sorted(c["dev|train"]) == sorted(c["dev|test"]) == DEV_IDS[t] and sorted(c["eval|train"]) == sorted(c["eval|test"]) == EVAL_IDS[t]),
           "training_clips_are_normal": bool(all("_train_normal_" in n_ for n_ in nm[tr]) and (lb[tr] == 0).all()),
           "training_clips_carry_their_domain": bool(np.isin(dom[tr], ["source", "target"]).all()),
           "source_clips_come_first_by_name": bool(all(v >= MIN_SOURCE_FIRST for v in first_target.values())),
           "every_training_section_has_target_clips": bool(all(v >= 0 for v in first_target.values())),
           "development_sections_have_both_kinds_in_both_domains": bool(both),
           "development_test_names_carry_label_and_domain": bool(all(("_test_normal_" in n_ or "_test_anomaly_" in n_) and ("_source_" in n_ or "_target_" in n_)
                                                                     for n_ in nm[(st == "dev") & (pt == "test")])),
           "evaluation_test_names_carry_no_label_or_domain": bool((not EVAL_IDS[t]) or (ev_test.any() and (dom[ev_test] == "").all()
                                                                  and not any(w in n_ for n_ in nm[ev_test] for w in ("normal", "anomaly", "source", "target")))),
           "evaluation_test_clips_are_numbered_consecutively": bool(numbered),
           "no_evaluation_label_in_the_store": bool((lb[ev_test] == -1).all()),
           "names_are_unique_in_each_folder": bool(all(len(set(nm[(st == s_) & (pt == p_)].tolist())) == int(((st == s_) & (pt == p_)).sum())
                                                       for s_ in ("dev", "eval") for p_ in ("train", "test"))),
           "clips": int(len(nm)), "skipped_archive_entries": int(len(m["skipped_archive_entries"])), "source_clips_before_the_first_target_clip": first_target,
           "shortest_clip_samples": int(min(int(k_) for d_ in m["lengths"].values() for k_ in d_)),
           "longest_clip_samples": int(max(int(k_) for d_ in m["lengths"].values() for k_ in d_)),
           "clips_repeated_to_fill_the_store": int(sum(c_ for d_ in m["lengths"].values() for k_, c_ in d_.items() if int(k_) < NS))}
    out["ok"] = bool(all(v for k, v in out.items() if isinstance(v, bool)) and out["shortest_clip_samples"] >= NS_MIN)
    return out


G0["b"] = {t: g0b(t) for t in TYPES}
print(f"G0(b) · the sections, the clip kinds, the domains and the names are the rules'; every clip is 16 kHz, one channel, at least {NS_MIN / SR:g} s:",
      all(v["ok"] for v in G0["b"].values()), {t: {k: v for k, v in d.items() if v is False} or "all checks true" for t, d in G0["b"].items()})
if not all(v["ok"] for v in G0["b"].values()):
    raise SystemExit(f"stop: G0(b) failed, the files are not what the rules assume: {G0['b']}")
N_TRAIN_ALL, N_TEST_ALL = (int(sum(sum(DCASE["counts"][t][f"{s_}|{p_}"].values()) for t in TYPES for s_ in ("dev", "eval"))) for p_ in ("train", "test"))
PREP_S = round(time.time() - t_data, 1)
print(f"data ready · {N_TRAIN_ALL} training clips and {N_TEST_ALL} test clips · {NS} samples per clip are kept ({NS / SR:g} s) · {PREP_S / 60:.1f} min")
print("sample formats of the WAV files, per type:", DCASE["formats"])
print("clips shorter than that, repeated from their start to fill it:", {t: G0["b"][t]["clips_repeated_to_fill_the_store"] for t in TYPES if G0["b"][t]["clips_repeated_to_fill_the_store"]} or "none")

In [ ]:
# 10 · Part 1 · The frozen recipe on these recordings, twice: as it is (a type is a machine: 15 classes), and with one class per type and
# attribute string (AT). ECHO on every clip, six trainings each (gate G1). Then the models leave the GPU.
# The training functions are notebook 23's whole block (one printed phrase names this data set), so they still hold its branches for
# the MIMII tracks and for the band views; with the DCASE lists (track "DC") and views=False those branches are never entered.
t_p1 = time.time()
ENCT = {"BEATs": "B", "BEATs-iter3": "B3", "EAT-base": "EB", "EAT-large": "EL"}
TRK = {"ARR": "ARR", "MIC+": "MICp", "MIC": "MIC", "DC": "DC"}


def n21_tag(track, flags, seed, fold):
    return f"{TRK[track]}_{'-'.join(f for f in ('XC', 'PL', 'TS', 'PD') if f in flags) or 'R0'}_s{seed}_f{fold}"


def n22_tag(ek, track, inp, flags, seed, fold):
    return f"{ENCT[ek]}_{TRK[track]}{'n' if inp == 'N2N-MWF' else ''}_{'-'.join(flags) or 'R0'}_s{seed}_f{fold}"


def stream_name(inp, fold):
    # the stream a model of fold k reads: denoised inputs use denoiser k, so no learned part has heard a new machine
    return {"MWF": "MWF_m6", "CH0": "CH0_m6", "N2N-CH0": f"N2N{fold}_m6", "N2N-MWF": f"N2NMWF{fold}_m6"}[inp]


def training_set(track, flags, fold):
    # items (type index, source 0 = MIMII / 1 = XC, row, class): one class per machine of the fold, and per extra machine with XC
    if track == "DC":
        return dcase_training_set()                                   # notebook 23: DCASE 2020's own lists (cell 13b)
    items, classes = [], []
    for ti, t in enumerate(TYPES):
        for p in FOLDS[fold]:
            if p in SPLIT[t]:
                c = len(classes); classes.append(f"{t}|{p}"); items.extend((ti, 0, int(r), c) for r in TRAIN_ROWS[(t, p)])
    if "XC" in flags:
        for ti, t in enumerate(TYPES):
            xm = np.array(XMETA[t]["mid"])
            for q in sorted(set(xm.tolist())):
                c = len(classes); classes.append(f"{t}|{q}|DCASE2020"); items.extend((ti, 1, int(r), c) for r in np.where(xm == q)[0])
    return np.array(items, dtype=np.int64), classes


def win(mm, row, s, n):
    return f32(mm[row, s:s + n])


def make_batch(track, inp, fold, items, ids, rng, n, want_teacher):
    # the student's crops; PD's teacher crops: the 6 dB twin of the same window through MWF (ARR only)
    B = len(ids); x = np.zeros((B, n), np.float32); tch = np.zeros((B, n), np.float32) if want_teacher else None
    has = np.zeros(B, bool)
    for b, it in enumerate(ids):
        ti, source, r, _ = items[it]; t = TYPES[ti]
        if source >= 2:                                               # DCASE 2020 (cell 13b): the clip itself is microphone 0
            mm = DST[t]
            x[b] = win(mm, r, int(rng.integers(0, mm.shape[1] - n + 1)), n)
            continue
        if source == 1:
            mm = XS[t]["CH0" if track == "MIC" else f"N2N{fold}"]
            x[b] = win(mm, r, int(rng.integers(0, mm.shape[1] - n + 1)), n)
            continue
        mm = ST[t][stream_name(inp, fold)]; s = int(rng.integers(0, mm.shape[1] - n + 1))
        x[b] = win(mm, r, s, n)
        if want_teacher:
            assert track == "ARR" and IS_TRAIN[t][r]
            tch[b] = win(ST[t]["MWF_6"], r, s, n); has[b] = True
    T = torch.from_numpy(tch).to(DEVICE) if want_teacher else None
    return torch.from_numpy(x).to(DEVICE), T, torch.from_numpy(has).to(DEVICE)


def fresh(enc, seed, n_classes):
    # LoRA back to zero, a new head and a new ArcFace layer: every training starts from the same frozen encoder
    torch.manual_seed(seed); enc.reset_lora()
    head = Head(enc.dim, CFG["emb_dim"]).to(DEVICE)
    arc = SubCenterArcFace(CFG["emb_dim"], n_classes, CFG["subcenters"], CFG["arc_scale"], CFG["arc_margin"]).to(DEVICE)
    return head, arc


def adapter_state(enc, head, arc):
    return {"lora": [{"A": w.A.detach().cpu(), "B": w.B.detach().cpu()} for w in enc.lora],
            "head": {k: v.cpu() for k, v in head.state_dict().items()}, "arc": {k: v.cpu() for k, v in arc.state_dict().items()}}


def load_lora(enc, lst):
    assert len(lst) == len(enc.lora), (len(lst), len(enc.lora))
    with torch.no_grad():
        for w, s in zip(enc.lora, lst):
            w.A.copy_(s["A"].to(DEVICE)); w.B.copy_(s["B"].to(DEVICE))


def teacher_tokens(enc, T):
    # the frozen encoder (LoRA off, evaluation mode, no gradient) on the teacher crops
    with torch.no_grad():
        LoRASwitch.on = False; enc.eval()
        try:
            return enc.tokens(T, amp=AMP)
        finally:
            LoRASwitch.on = True; enc.train()


def step_losses(enc, arc, head, X, y, mix, lam, perm, T, has, n):
    tok = enc.tokens(X, amp=AMP)
    cos = arc.cosine(head(enc.pool(tok, n)))
    ce = lam * arc.loss(cos, y) + (1.0 - lam) * arc.loss(cos, y[perm]) if mix else arc.loss(cos, y)
    pd = None
    if T is not None and bool(has.any()):
        tt = teacher_tokens(enc, T[has])
        pd = (1.0 - F.cosine_similarity(tok[has], tt, dim=-1)).mean()
    return ce, pd


def train(ek, track, inp, flags, seed, fold):
    # notebook 21's training for any encoder: LoRA + head + sub-centre ArcFace, mixup, warm-up and cosine decay, PD's λ rule
    enc = ENC[ek]; tag = n22_tag(ek, track, inp, flags, seed, fold)
    final, resume = OUT / f"ft_{tag}.pt", OUT / f"ft_{tag}_resume.pt"
    items, classes = training_set(track, flags, fold)
    head, arc = fresh(enc, seed, len(classes))
    if final.exists():
        st = torch.load(final, map_location="cpu", weights_only=False)
        assert st["classes"] == classes, "the saved model used other classes"
        load_lora(enc, st["lora"]); head.load_state_dict(st["head"]); arc.load_state_dict(st["arc"])
        return head, st["log"]
    pd_on = "PD" in flags
    params = lora_params_of(enc) + list(head.parameters()) + list(arc.parameters())
    opt = torch.optim.AdamW([{"params": lora_params_of(enc), "lr": CFG["lr_lora"], "weight_decay": 0.0},
                             {"params": list(head.parameters()) + list(arc.parameters()), "lr": CFG["lr_head"], "weight_decay": CFG["wd_head"]}])
    spe = len(items) // CFG["batch"]
    total, warm = spe * CFG["epochs"], spe * CFG["warmup_epochs"]
    lr_f = lambda s: (s + 1) / warm if s < warm else 0.5 * (1.0 + math.cos(math.pi * (s - warm) / max(1, total - warm)))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, [lr_f, lr_f])
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    log = {"tag": tag, "encoder": ek, "track": track, "input": inp, "flags": list(flags), "seed": seed, "fold": fold, "cfg": CFG,
           "classes": classes, "n_train": int(len(items)), "n_xc": int(np.isin(items[:, 1], (1, 3)).sum()), "lambda_pd": None,
           "epochs": [], "resumed_at_epoch": []}
    start, n = 0, CROP
    if resume.exists():
        st = torch.load(resume, map_location="cpu", weights_only=False)
        load_lora(enc, st["lora"]); head.load_state_dict(st["head"]); arc.load_state_dict(st["arc"])
        opt.load_state_dict(st["opt"]); sched.load_state_dict(st["sched"]); scaler.load_state_dict(st["scaler"])
        log, start = st["log"], st["epoch"] + 1
        log["resumed_at_epoch"].append(start); print(f"resuming {tag} at epoch {start}")
    if pd_on and log["lambda_pd"] is None:
        # the rules' balance: λ_PD = mean ArcFace loss / mean distillation loss over a few batches, before any update
        rng = np.random.default_rng([seed, fold, ALL_TRACKS.index(track), 999, ENC_ORDER.index(ek)]); ce_l, pd_l = [], []
        enc.train(); head.train(); arc.train()
        with torch.no_grad():
            for _ in range(CFG["pd_balance_batches"]):
                ids = rng.choice(len(items), CFG["batch"], replace=False)
                X, T, has = make_batch(track, inp, fold, items, ids, rng, n, True)
                y = torch.from_numpy(items[ids, 3]).to(DEVICE)
                ce, pd = step_losses(enc, arc, head, X, y, False, 1.0, None, T, has, n)
                ce_l.append(float(ce)); pd_l.append(float(pd)) if pd is not None else None
        log["lambda_pd"] = round(float(np.mean(ce_l) / max(np.mean(pd_l), 1e-6)), 4)
        print(f"{tag}: λ_PD = {log['lambda_pd']} (ArcFace {np.mean(ce_l):.3f}, distillation {np.mean(pd_l):.4f} at the start)")
    print(f"{tag}: {len(items)} training clips ({log['n_xc']} from DCASE 2025's evaluation types) · {len(classes)} classes · {spe} steps per epoch", flush=True)
    for ep in range(start, CFG["epochs"]):
        rng = np.random.default_rng([seed, fold, ALL_TRACKS.index(track), ep, ENC_ORDER.index(ek)])
        np.random.seed(seed * 1000 + ep)                              # BEATs draws its layer-drop from numpy's global stream
        order = rng.permutation(len(items))
        enc.train(); head.train(); arc.train()
        t0, tot, tce, tpd, n_mix = time.time(), 0.0, 0.0, 0.0, 0
        for b in range(spe):
            ids = order[b * CFG["batch"]:(b + 1) * CFG["batch"]]
            mix = bool(rng.random() < CFG["mixup_prob"])
            X, T, has = make_batch(track, inp, fold, items, ids, rng, n, pd_on)
            y = torch.from_numpy(items[ids, 3]).to(DEVICE)
            lam, perm = 1.0, None
            if mix:
                lam = float(rng.beta(CFG["mixup_beta"], CFG["mixup_beta"]))
                perm = torch.from_numpy(rng.permutation(len(ids))).to(DEVICE)
                X = lam * X + (1.0 - lam) * X[perm]
                if pd_on:
                    T = lam * T + (1.0 - lam) * T[perm]; has = has & has[perm]
            ce, pd = step_losses(enc, arc, head, X, y, mix, lam, perm, T, has, n)
            loss = ce if pd is None else ce + log["lambda_pd"] * pd
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, CFG["clip_norm"])
            scaler.step(opt); scaler.update(); sched.step()
            tot += float(loss.detach()); tce += float(ce.detach()); tpd += 0.0 if pd is None else float(pd.detach()); n_mix += mix
        log["epochs"].append({"epoch": ep, "mean_loss": round(tot / spe, 4), "mean_arcface": round(tce / spe, 4),
                              "mean_distillation": round(tpd / spe, 5), "mixup_batches": int(n_mix), "seconds": round(time.time() - t0, 1)})
        print(f"{tag} · epoch {ep + 1}/{CFG['epochs']} · loss {tot / spe:.3f}" + (f" (distillation {tpd / spe:.4f})" if pd_on else "")
              + f" · {time.time() - t0:.0f} s", flush=True)
        torch.save({**adapter_state(enc, head, arc), "opt": opt.state_dict(), "sched": sched.state_dict(),
                    "scaler": scaler.state_dict(), "epoch": ep, "log": log}, resume)
    enc.eval(); head.eval(); arc.eval()
    torch.save({**adapter_state(enc, head, arc), "classes": classes, "log": log}, final)
    resume.unlink(missing_ok=True)
    return head, log


@torch.no_grad()
def run_pass(enc, mm, head=None, rows=None, views=True):
    # whole clips through the encoder as it stands (LoRA as switched): per clip the global mean P, F = head(P) when a head is
    # given, and the band views: per frequency row, the mean, the max and RDP (γ = 4) over the time patches
    rows = np.arange(mm.shape[0]) if rows is None else rows
    n, bs = mm.shape[1], CFG["embed_batch"]; enc.eval()
    P = np.zeros((len(rows), enc.dim), np.float16); Fh = np.zeros((len(rows), CFG["emb_dim"]), np.float16) if head is not None else None
    V = {v: np.zeros((len(rows), 8, enc.dim), np.float16) for v in VIEWS} if views else None
    for i in range(0, len(rows), bs):
        x = torch.from_numpy(f32(mm[rows[i:i + bs]])).to(DEVICE); m_ = len(x)
        g = enc.grid(enc.tokens(x, amp=AMP), n)
        p = g.mean((1, 2)); P[i:i + m_] = p.cpu().numpy()
        if head is not None:
            Fh[i:i + m_] = head(p).cpu().numpy()
        if views:
            V["mean"][i:i + m_] = g.mean(1).cpu().numpy(); V["max"][i:i + m_] = g.amax(1).cpu().numpy()
            d = (g - g.mean(1, keepdim=True)).norm(dim=-1)            # RDP (arXiv 2603.04605): deviation from the band's mean
            dh = d / d.amax(1, keepdim=True).clamp_min(1e-12)
            w = (1.0 + dh) ** RDP_GAMMA; w = w / w.sum(1, keepdim=True)
            V[f"rdp{RDP_GAMMA}"][i:i + m_] = (w.unsqueeze(-1) * g).sum(1).cpu().numpy()
    return P, Fh, V


LOGS, T_TRAIN = {}, {}
DC_DIR = OUT / "dcase"; DC_DIR.mkdir(exist_ok=True)
ECHO_DC = {}
for t in TYPES:
    f = DC_DIR / f"echo_{t}.npy"
    if not f.exists():
        tmp = DC_DIR / f"echo_{t}.part.npy"; np.save(tmp, echo_stream(DST[t]).astype(np.float16)); os.replace(tmp, f)
    ECHO_DC[t] = np.load(f)
T_ECHO = round(time.time() - t_p1, 1)
print(f"ECHO on every clip: {({t: list(ECHO_DC[t].shape) for t in TYPES})} · {T_ECHO / 60:.1f} min in this session", flush=True)


CLASS_MODE = "type"                                                   # "type": a type is a machine; "attribute": one class per type and attribute string
OUT_AT = OUT / "attribute_classes"; OUT_AT.mkdir(exist_ok=True)       # the second set of trainings and embeddings


def attribute_of(name):
    # what follows the clip's number in a training file name: section_00_source_train_normal_0001_<attribute>.wav
    m_ = re.match(r"section_\d+_(?:source|target)_train_normal_\d+_(.+)\.wav$", name)
    return m_.group(1) if m_ else "noAttribute"


class variant:
    # the same functions on the second set of trainings: the folder they read and write, the class lists, the logs, and the cache of
    # embeddings. Everything is put back on leaving.
    def __init__(self, mode):
        self.mode = mode

    def __enter__(self):
        g = globals(); self.keep = (g["OUT"], g["CLASS_MODE"], g["LOGS"], g["T_TRAIN"])
        g["OUT"], g["CLASS_MODE"], g["LOGS"], g["T_TRAIN"] = OUT_AT, self.mode, {}, {}
        g.get("FM", {}).clear()
        return self

    def __exit__(self, *a):
        g = globals(); self.logs, self.times = g["LOGS"], g["T_TRAIN"]
        g["OUT"], g["CLASS_MODE"], g["LOGS"], g["T_TRAIN"] = self.keep
        g.get("FM", {}).clear()


def dcase_training_set():
    # DCASE 2025's own lists (a type is a machine): each type's first CFG["xc_cap"] training clips by name, as notebook 28 took an evaluation
    # section's. With CLASS_MODE "attribute", one class per type and attribute string; items (type index, source 2 = development type /
    # 3 = evaluation type, row, class)
    items, classes = [], []
    for ti, t in enumerate(TYPES):
        nm = np.array(DMETA[t]["name"])
        for set_, src_, cap in (("dev", 2, CFG["xc_cap"]), ("eval", 3, CFG["xc_cap"])):
            mids = np.array(DMETA[t]["mid"])
            for q in sorted(set(mids[dc_rows(t, set_, "train")].tolist())):
                rows = dc_rows(t, set_, "train", q); rows = rows[np.argsort(nm[rows], kind="stable")]
                rows = rows[:cap] if cap else rows
                what = 'development' if set_ == 'dev' else 'additional'
                if CLASS_MODE == "attribute":
                    att = [attribute_of(nm[r]) for r in rows]
                    for a in sorted(set(att)):
                        c = len(classes); classes.append(f"{t}|{q}|DCASE2025-{what}|{a}")
                        items.extend((ti, src_, int(r), c) for r, a_ in zip(rows, att) if a_ == a)
                else:
                    c = len(classes); classes.append(f"{t}|{q}|DCASE2025-{what}")
                    items.extend((ti, src_, int(r), c) for r in rows)
    return np.array(items, dtype=np.int64), classes


_it, _cl = dcase_training_set()
print(f"DCASE 2025 training lists: {len(_it)} clips, {len(_cl)} classes ({int((_it[:, 1] == 3).sum())} clips of the evaluation types) · "
      f"one training takes about {len(_it) / CFG['batch'] * CFG['epochs'] * ENC['BEATs'].step_s * 1.15 / 60:.0f} min on this GPU")
F_CLASSES, F_TRAIN = list(_cl), int(len(_it))
assert len(F_CLASSES) == REF_CLASSES and F_CLASSES == [f"{t}|{q}|DCASE2025-{w}" for t in TYPES for w, ids in (("development", DEV_IDS[t]), ("additional", EVAL_IDS[t]))
                                                       for q in ids], "stop: the classes are not the rules' types"
assert all(int((_it[:, 3] == c).sum()) <= FROZEN_CAP for c in range(len(F_CLASSES))), "more than the cap from a type"
# what the rules say of the clips in this list: they are source-domain clips (file names put the source domain first)
F_ADD_DOMAINS = dict(Counter(DMETA[TYPES[int(ti)]]["domain"][int(r)] for ti, src_, r, c in _it))
print("the frozen recipe's training clips, by domain:", F_ADD_DOMAINS)
with variant("attribute"):
    _it_at, AT_CLASSES = dcase_training_set()
assert sorted(map(tuple, _it_at[:, :3].tolist())) == sorted(map(tuple, _it[:, :3].tolist())), "stop: the attribute classes are not on the same clips"
AT_PER_TYPE = {t: sum(1 for c in AT_CLASSES if c.startswith(t + "|")) for t in TYPES}
print(f"attribute classes (AT): {len(AT_CLASSES)} classes on the same {len(_it_at)} clips · classes per type: {AT_PER_TYPE}")
print(f"the twelve trainings together: about {12 * len(_it) / CFG['batch'] * CFG['epochs'] * ENC['BEATs'].step_s * 1.15 / 60:.0f} min on this GPU, "
      f"and a pass over {sum(len(DMETA[t]['name']) for t in TYPES)} clips after each", flush=True)


def ensure_dc(ek, seed):
    # one model of MIC's recipe on DCASE's lists (no folds), then F and P of every DCASE clip; gate G1
    tag = n22_tag(ek, "DC", "CH0", ["XC"], seed, 0); fin, ef = OUT / f"ft_{tag}.pt", OUT / f"emb_{tag}.npz"
    if not (fin.exists() and ef.exists()):
        t0 = time.time()
        head, _ = train(ek, "DC", "CH0", ["XC"], seed, 0)
        LoRASwitch.on = True; head.eval(); out = {}
        for t in TYPES:
            P, Fh, _ = run_pass(ENC[ek], DST[t], head, views=False)
            out[f"P__{t}"], out[f"F__{t}"] = P, Fh
        tmp = OUT / f"emb_{tag}.part.npz"
        np.savez(tmp, **out, meta=json.dumps({"tag": tag, "weights_md5": md5(fin), "encoder": ek, "test_run": TEST, "date": time.strftime("%Y-%m-%d %H:%M")}))
        os.replace(tmp, ef)
        T_TRAIN[tag] = round(time.time() - t0, 1)
        print(f"saved {ef.name} · {T_TRAIN[tag] / 60:.1f} min for training and the pass", flush=True)
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    log = torch.load(fin, map_location="cpu", weights_only=False)["log"]; LOGS[tag] = log
    return bool(log["epochs"][-1]["mean_loss"] < log["epochs"][0]["mean_loss"])


# the six trainings of the frozen recipe (finished ones are read from Drive; an unfinished one continues from its last epoch)
DC_G1 = {}
for ek, _, seeds in FROZEN:
    for s in seeds:
        DC_G1[f"{ek}|seed {s}"] = ensure_dc(ek, s)
G1_F = {k: {"first_epoch_loss": LOGS[k]["epochs"][0]["mean_loss"], "last_epoch_loss": LOGS[k]["epochs"][-1]["mean_loss"], "epochs": len(LOGS[k]["epochs"]),
            "seconds": [e["seconds"] for e in LOGS[k]["epochs"]], "resumed_at_epoch": LOGS[k]["resumed_at_epoch"]} for k in LOGS}
print("G1 · the training loss fell in each of the six trainings of the frozen recipe:", all(DC_G1.values()), DC_G1)
if not TEST and not all(DC_G1.values()):
    raise SystemExit("stop: G1 failed for a training of the frozen recipe")
assert all(LOGS[k]["classes"] == F_CLASSES and LOGS[k]["cfg"] == CFG for k in LOGS), "stop: a saved training used other classes or settings"

# the same six trainings with attribute classes, in their own folder (finished ones are read from Drive, an unfinished one continues)
with variant("attribute") as _v:
    DC_G1_AT = {f"{ek}|seed {s}": ensure_dc(ek, s) for ek, _, seeds in FROZEN for s in seeds}
LOGS_AT, T_TRAIN_AT = _v.logs, _v.times
G1_AT = {k: {"first_epoch_loss": LOGS_AT[k]["epochs"][0]["mean_loss"], "last_epoch_loss": LOGS_AT[k]["epochs"][-1]["mean_loss"], "epochs": len(LOGS_AT[k]["epochs"]),
             "seconds": [e["seconds"] for e in LOGS_AT[k]["epochs"]], "resumed_at_epoch": LOGS_AT[k]["resumed_at_epoch"]} for k in LOGS_AT}
print("G1 · the training loss fell in each of the six trainings with attribute classes:", all(DC_G1_AT.values()), DC_G1_AT)
if not TEST and not all(DC_G1_AT.values()):
    raise SystemExit("stop: G1 failed for a training with attribute classes")
assert list(LOGS_AT) == list(LOGS) and all(LOGS_AT[k]["classes"] == AT_CLASSES and LOGS_AT[k]["cfg"] == CFG for k in LOGS_AT), \
    "stop: a saved training with attribute classes used other classes or settings"

# Part 1 is on disk (ft_*.pt, emb_*.npz, dcase/echo_*.npy, and the same in attribute_classes/). Its models leave the GPU before the view is trained
ENC_INFO = {k: {**e.info, "name": e.name, "layout": e.layout, "step_seconds": e.step_s} for k, e in ENC.items()}
LOGS_F, T_TRAIN_F = LOGS, T_TRAIN
ENC.clear()
for _n in ("model", "chk", "LORA", "ECHO", "enc", "x4", "Z4", "ref"):
    globals().pop(_n, None)
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()
T_PART1 = round(time.time() - t_p1, 1)
print(f"Part 1 done · {T_PART1 / 60:.0f} min in this cell · {(time.time() - T_START) / 60:.0f} min in this session", flush=True)

In [ ]:
# 11 · The frozen system from Part 1's saved files (notebook 25's code, reading this notebook's folder): MIC's scorer, and the spread
# σ_F of its log score on normal clips its trainings did not use, for both sets of trainings. No label is read in this cell.
def unit(E):
    return E / np.linalg.norm(E, axis=-1, keepdims=True)


def b1(R, X):
    return 1.0 - (unit(X) @ unit(R).T).max(1)


def ldn1(R, X):
    # Wilkinghoff et al. (TASLP 2026), ratio form with K = 1: min over references y of d(x, y) / d(y, y₁)
    U = unit(R); Srr = U @ U.T; np.fill_diagonal(Srr, -np.inf)
    d1 = np.maximum(1.0 - Srr.max(1), LDN_FLOOR)
    return ((1.0 - unit(X) @ U.T) / d1[None, :]).min(1)


def dc_metrics(y, s):
    # as the official evaluator: sklearn's AUC, and its pAUC with max_fpr = 0.1 (standardised)
    return {"AUC": float(roc_auc_score(y, s)), "pAUC": float(roc_auc_score(y, s, max_fpr=0.1)), "n_test": int(len(y)), "n_anomalies": int(np.sum(y))}


_missing = [f for f in [f"emb_{c}_DC_XC_s{s}_f0.npz" for _, c, sd in FROZEN for s in sd] + [f"dcase/echo_{t}.npy" for t in TYPES] if not (OUT / f).exists()]
if _missing:
    raise SystemExit(f"stop: Part 1's files are not all in {OUT} (for example {_missing[0]})")
assert all(len(ECHO_DC[t]) == len(DMETA[t]["name"]) for t in TYPES), "stop: ECHO's saved embeddings do not have this session's number of clips"
FM = {}


def frozen_matrix(code, seeds, t):
    k = (code, t)
    if k not in FM:
        Es = [np.load(OUT / f"emb_{code}_DC_XC_s{s}_f0.npz", allow_pickle=False)[f"F__{t}"].astype(np.float32) for s in seeds]
        E = np.concatenate([unit(e) for e in Es], 1) / math.sqrt(len(Es))
        FM[k] = np.concatenate([E, unit(ECHO_DC[t].astype(np.float64))], 1) / math.sqrt(2.0)
        assert len(FM[k]) == len(DMETA[t]["name"]), f"Part 1's embeddings of {t} do not have this session's number of clips"
    return FM[k]


def machines(t, set_):
    return sorted(set(np.array(DMETA[t]["mid"])[dc_rows(t, set_, "test")].tolist()))


def frozen_score(t, ref, x):
    return sum(ldn1(frozen_matrix(c, sd, t)[ref], frozen_matrix(c, sd, t)[x]) for _, c, sd in FROZEN) / len(FROZEN)


def spread(logs):
    # the centre and the robust spread (1.4826 × the median absolute deviation) of a system's log scores on normal clips
    if len(logs) == 0:
        return {"n": 0, "median": None, "sigma": None}
    med = float(np.median(logs))
    return {"n": int(len(logs)), "median": med, "sigma": float(1.4826 * np.median(np.abs(logs - med)))}


def part_score(code, t, ref, x):
    # one encoder's part of MIC's scorer on its own: its seeds' F with ECHO concatenated (the frozen system is the mean of the two parts)
    sd = next(s for _, c, s in FROZEN if c == code)
    return ldn1(frozen_matrix(code, sd, t)[ref], frozen_matrix(code, sd, t)[x])


def echo_score(t, ref, x):
    # ECHO alone: global LDN1 on its frozen embedding
    E = ECHO_DC[t].astype(np.float64)
    return ldn1(E[ref], E[x])


SF = {}                                                               # (type, set, machine) → the frozen system's scores of its test clips


def frozen_of(t, set_, q):
    # computed once per machine; the references are all of the machine's normal training clips
    k = (t, set_, q)
    if k not in SF:
        SF[k] = frozen_score(t, dc_rows(t, set_, "train", q), dc_rows(t, set_, "test", q))
    return SF[k]


_R, _X = np.random.default_rng(11).standard_normal((40, 9)), np.random.default_rng(12).standard_normal((7, 9))
assert np.allclose(ldn1(_R, _X), [min((1 - unit(x) @ unit(r)) / max(1 - max(unit(r) @ unit(r2) for j, r2 in enumerate(_R) if j != i), LDN_FLOOR)
                                      for i, r in enumerate(_R)) for x in _X]), "stop: LDN1 is not the rules' ratio"

t_f = time.time()
LAB = {t: np.array(DMETA[t]["label"]) for t in TYPES}


def frozen_spread(t):
    # σ_F (notebook 25's rule): the frozen system's log score on the training clips that its trainings did not use (those after the
    # first FROZEN_CAP of a type, by name), against the clips they did use. No test clip.
    nm_, logs, kept = np.array(DMETA[t]["name"]), [], []
    for set_ in ("dev", "eval"):
        for q in machines(t, set_):
            rows = dc_rows(t, set_, "train", q); rows = rows[np.argsort(nm_[rows], kind="stable")]
            if len(rows) > FROZEN_CAP:
                s_ = frozen_score(t, rows[:FROZEN_CAP], rows[FROZEN_CAP:])
                logs.append(np.log(np.maximum(s_, LDN_FLOOR))); kept.extend(zip(nm_[rows[FROZEN_CAP:]].tolist(), s_.tolist()))
    return spread(np.concatenate(logs) if logs else np.zeros(0)), kept


CAL_F, CALF_SC, CAL_F_AT, CALF_SC_AT, SF_AT = {}, {}, {}, {}, {}
for t in TYPES:
    CAL_F[t], CALF_SC[t] = frozen_spread(t)
    FM.clear(); gc.collect()
print("form ZP · the frozen system's spread on normal clips it did not train on (clips, σ of the log score):",
      {t: (CAL_F[t]["n"], None if CAL_F[t]["sigma"] is None else round(CAL_F[t]["sigma"], 4)) for t in TYPES}, f"· {time.time() - t_f:.0f} s")
# the same from the trainings with attribute classes: the spread, and the score of every test clip (no label is read; the references
# are all of the type's training clips, as for the system as it is)
_missing = [f for f in [f"emb_{c}_DC_XC_s{s}_f0.npz" for _, c, sd in FROZEN for s in sd] if not (OUT_AT / f).exists()]
if _missing:
    raise SystemExit(f"stop: the files of the trainings with attribute classes are not all in {OUT_AT} (for example {_missing[0]})")
with variant("attribute"):
    for t in TYPES:
        CAL_F_AT[t], CALF_SC_AT[t] = frozen_spread(t)
        for set_ in ("dev", "eval"):
            for q in machines(t, set_):
                SF_AT[(t, set_, q)] = frozen_score(t, dc_rows(t, set_, "train", q), dc_rows(t, set_, "test", q))
        FM.clear(); gc.collect()
print("the same with attribute classes (clips, σ):", {t: (CAL_F_AT[t]["n"], None if CAL_F_AT[t]["sigma"] is None else round(CAL_F_AT[t]["sigma"], 4)) for t in TYPES},
      f"· {time.time() - t_f:.0f} s")
if DEVICE == "cuda":
    torch.backends.cudnn.benchmark = True                             # as notebook 25's setup: the view's inputs have one size

In [ ]:
# 12 · Part 2 · The second view (notebook 25's cell): TASTgram-MFN trained with Noisy-ArcMix (Choi and Choi, ICASSP 2024, arXiv 2310.06364)
#
# The model classes, the log-mel settings and the Noisy-ArcMix step are adapted from https://github.com/soonhyeon/Noisy-ArcMix at
# commit 674c759bfaa7723619a4b1d8d00eb9d155c50ba6 (model/net.py, losses.py, dataloader.py, utils.py, trainer.py). Its model file
# says: "modification made on the basis of link:https://github.com/liuyoude/STgram-MFN". Its licence:
#
#   MIT License
#
#   Copyright (c) 2023 soonhyeon
#
#   Permission is hereby granted, free of charge, to any person obtaining a copy
#   of this software and associated documentation files (the "Software"), to deal
#   in the Software without restriction, including without limitation the rights
#   to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
#   copies of the Software, and to permit persons to whom the Software is
#   furnished to do so, subject to the following conditions:
#
#   The above copyright notice and this permission notice shall be included in all
#   copies or substantial portions of the Software.
#
#   THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
#   IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
#   FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
#   AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
#   LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
#   OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
#   SOFTWARE.
#
# Changed here (the rules list every deviation): the number of classes; the frame count and the last kernel follow the clip length
# (313 and 8 × 20 for DCASE's 10 s clips, as in the reference); mixed precision with a gradient scaler, and the ArcFace layer in
# float32 with a floor under its square root; batches come from one int16 tensor and the log-mel is computed on the device; the
# seeds, the hold-out draw, saving and resuming are this notebook's. The reference's unused options (sub-centres, easy margin,
# the other training modes) are left out.
class Bottleneck(nn.Module):
    def __init__(self, inp, oup, stride, expansion):
        super(Bottleneck, self).__init__()
        self.connect = stride == 1 and inp == oup
        self.conv = nn.Sequential(
            # pw
            nn.Conv2d(inp, inp * expansion, 1, 1, 0, bias=False),
            nn.BatchNorm2d(inp * expansion),
            nn.PReLU(inp * expansion),
            # dw
            nn.Conv2d(inp * expansion, inp * expansion, 3, stride, 1, groups=inp * expansion, bias=False),
            nn.BatchNorm2d(inp * expansion),
            nn.PReLU(inp * expansion),
            # pw-linear
            nn.Conv2d(inp * expansion, oup, 1, 1, 0, bias=False),
            nn.BatchNorm2d(oup),
        )

    def forward(self, x):
        if self.connect:
            return x + self.conv(x)
        else:
            return self.conv(x)


class ConvBlock(nn.Module):
    def __init__(self, inp, oup, k, s, p, dw=False, linear=False):
        super(ConvBlock, self).__init__()
        self.linear = linear
        if dw:
            self.conv = nn.Conv2d(inp, oup, k, s, p, groups=inp, bias=False)
        else:
            self.conv = nn.Conv2d(inp, oup, k, s, p, bias=False)
        self.bn = nn.BatchNorm2d(oup)
        if not linear:
            self.prelu = nn.PReLU(oup)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        if self.linear:
            return x
        else:
            return self.prelu(x)


Mobilefacenet_bottleneck_setting = [
    # t, c , n ,s
    [2, 128, 2, 2],
    [4, 128, 2, 2],
    [4, 128, 2, 2],
]


class MobileFaceNet(nn.Module):
    def __init__(self, num_class, last_kernel, bottleneck_setting=Mobilefacenet_bottleneck_setting):
        super(MobileFaceNet, self).__init__()
        self.conv1 = ConvBlock(3, 64, 3, 2, 1)
        self.dw_conv1 = ConvBlock(64, 64, 3, 1, 1, dw=True)
        self.inplanes = 64
        block = Bottleneck
        self.blocks = self._make_layer(block, bottleneck_setting)
        self.conv2 = ConvBlock(bottleneck_setting[-1][1], 512, 1, 1, 0)
        self.linear7 = ConvBlock(512, 512, last_kernel, 1, 0, dw=True, linear=True)   # (8, 20) in the reference: 128 × 313 → 8 × 20
        self.linear1 = ConvBlock(512, 128, 1, 1, 0, linear=True)
        self.fc_out = nn.Linear(128, num_class)                       # the reference has it too; with ArcFace on, its output is unused
        # init
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                n = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
                m.weight.data.normal_(0, math.sqrt(2. / n))
            elif isinstance(m, nn.BatchNorm2d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()

    def _make_layer(self, block, setting):
        layers = []
        for t, c, n, s in setting:
            for i in range(n):
                if i == 0:
                    layers.append(block(self.inplanes, c, s, t))
                else:
                    layers.append(block(self.inplanes, c, 1, t))
                self.inplanes = c
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.dw_conv1(x)
        x = self.blocks(x)
        x = self.conv2(x)
        x = self.linear7(x)
        x = self.linear1(x)
        return x.reshape(x.size(0), -1)                               # the 128-d feature


class TgramNet(nn.Module):
    def __init__(self, n_frames, num_layer=3, mel_bins=128, win_len=1024, hop_len=512):
        super(TgramNet, self).__init__()
        # if "center=True" of stft, padding = win_len / 2
        self.conv_extrctor = nn.Conv1d(1, mel_bins, win_len, hop_len, win_len // 2, bias=False)
        self.conv_encoder = nn.Sequential(
            *[nn.Sequential(
                nn.LayerNorm(n_frames),                               # 313 in the reference (10 s)
                nn.LeakyReLU(0.2, inplace=True),
                nn.Conv1d(mel_bins, mel_bins, 3, 1, 1, bias=False),
            ) for _ in range(num_layer)])

    def forward(self, x):
        out = self.conv_extrctor(x)
        out = self.conv_encoder(out)
        return out


class Temporal_Attention(nn.Module):
    def __init__(self, feature_dim=128):
        super().__init__()
        self.feature_dim = feature_dim
        self.max_pool = nn.AdaptiveMaxPool1d(1)
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x: (B, 1, 128, 313)
        x = x.squeeze(1)
        x = x.transpose(1, 2)  # (B, 313, 128)
        x1 = self.max_pool(x)  # (B, 313, 1)
        x2 = self.avg_pool(x)  # (B, 313, 1)
        feats = x1 + x2
        feats = feats.repeat(1, 1, self.feature_dim)
        refined_feats = self.sigmoid(feats).transpose(1, 2) * x.transpose(1, 2)
        return refined_feats


class ArcMarginProduct(nn.Module):
    def __init__(self, in_features=128, out_features=200, s=30.0, m=0.7):
        super(ArcMarginProduct, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.s = s
        self.m = m
        self.weight = Parameter(torch.Tensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        # make the function cos(theta+m) monotonic decreasing while theta in [0°,180°]
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def forward(self, x, label):
        cosine = F.linear(F.normalize(x), F.normalize(self.weight))
        sine = torch.sqrt((1.0 - torch.pow(cosine, 2)).clamp_min(SV["sine_floor"]))   # the floor is this notebook's (a deviation)
        phi = cosine * self.cos_m - sine * self.sin_m
        phi = torch.where((cosine - self.th) > 0, phi, cosine - self.mm)
        one_hot = torch.zeros(cosine.size(), device=x.device)
        one_hot.scatter_(1, label.view(-1, 1).long(), 1)
        output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        output = output * self.s
        return output


MEMFMT = torch.contiguous_format                                      # set by pick_memory_format (a speed setting only)


class TASTgramMFN(nn.Module):
    def __init__(self, num_classes, n_frames, last_kernel, c_dim=128, win_len=1024, hop_len=512,
                 bottleneck_setting=Mobilefacenet_bottleneck_setting, m=0.7, s=30):
        super().__init__()
        self.arcface = ArcMarginProduct(in_features=c_dim, out_features=num_classes, m=m, s=s)
        self.tgramnet = TgramNet(n_frames, mel_bins=c_dim, win_len=win_len, hop_len=hop_len)
        self.mobilefacenet = MobileFaceNet(num_class=num_classes, last_kernel=last_kernel, bottleneck_setting=bottleneck_setting)
        self.temporal_attention = Temporal_Attention(feature_dim=c_dim)

    def forward(self, x_wav, x_mel, label):
        # the reference's "noisy_arcmix" mode: the margin goes to `label` (in training, the first clip's class)
        with torch.autocast(device_type=DEVICE, enabled=AMP):
            x_t = self.tgramnet(x_wav).unsqueeze(1)
            x_mel_temp_att = self.temporal_attention(x_mel).unsqueeze(1)
            x = torch.cat((x_t.float(), x_mel, x_mel_temp_att), dim=1)   # .float(): the three channels in one type under autocast
            feature = self.mobilefacenet(x.contiguous(memory_format=MEMFMT))
        feature = feature.float()
        with torch.autocast(device_type=DEVICE, enabled=False):      # ArcFace in float32 (a deviation)
            out = self.arcface(feature, label)
        return out, feature


MEL = torchaudio.transforms.MelSpectrogram(sample_rate=SR, n_fft=SV["n_fft"], hop_length=SV["hop"], n_mels=SV["n_mels"], power=2,
                                           pad_mode="constant", norm="slaney", mel_scale="slaney").to(DEVICE)
CE = nn.CrossEntropyLoss()


@torch.no_grad()
def log_mel(x):
    # (B, samples) float32 in [−1, 1] → (B, 1, mel bands, frames): the reference's file_to_log_mel_spectrogram with power = 2
    return (20.0 / 2 * torch.log10(MEL(x) + sys.float_info.epsilon)).unsqueeze(1)


def frames_of(ns):
    # the frames of an STFT with center=True, which is also the length of the Tgram convolution (kernel 1024, stride 512, padding 512)
    return 1 + ns // SV["hop"]


def last_kernel(n_mels, n_frames):
    # the feature map after the network's four stride-2 stages (3 × 3, padding 1): 128 × 313 → 8 × 20
    h, w = n_mels, n_frames
    for _ in range(4):
        h, w = (h - 1) // 2 + 1, (w - 1) // 2 + 1
    return (h, w)


def training_lists():
    # one class per machine: for each type, its development machines (all train/ clips of the development zip), then its evaluation
    # machines (all clips of the additional training zip); items (type index, row in the type's store, class)
    items, classes, cls_of = [], [], {}
    for ti, t in enumerate(TYPES):
        nm, mids = np.array(DMETA[t]["name"]), np.array(DMETA[t]["mid"])
        for set_, what in (("dev", "development"), ("eval", "additional")):
            for q in sorted(set(mids[dc_rows(t, set_, "train")].tolist())):
                rows = dc_rows(t, set_, "train", q); rows = rows[np.argsort(nm[rows], kind="stable")]
                c = len(classes); classes.append(f"{t}|{q}|DCASE2025-{what}"); cls_of[(t, q)] = c
                items.extend((ti, int(r), c) for r in rows)
    return np.array(items, dtype=np.int64), classes, cls_of


AUDIO, LABELS, STORE_DEV, MF_TIMES = None, None, None, {}


def build_audio():
    # every training-list clip in one int16 tensor: on the GPU when it has room (batches are then gathered without the CPU), else in RAM
    global AUDIO, LABELS, STORE_DEV
    if AUDIO is not None:
        return
    n = len(ITEMS); need = n * NS * 2 / 1e9; dev = DEVICE; t0 = time.time()
    if DEVICE == "cuda" and torch.cuda.mem_get_info()[0] / 1e9 < need + SV["gpu_headroom_gb"]:
        dev = "cpu"
    A = torch.empty((n, NS), dtype=torch.int16, device=dev)
    for ti, t in enumerate(TYPES):
        pos = np.where(ITEMS[:, 0] == ti)[0]
        for i in range(0, len(pos), 256):
            p = pos[i:i + 256]
            A[torch.from_numpy(p).to(dev)] = torch.from_numpy(np.ascontiguousarray(DST[t][ITEMS[p, 1]])).to(dev)
    AUDIO, STORE_DEV, LABELS = A, dev, torch.from_numpy(ITEMS[:, 2].copy()).to(dev)
    print(f"training audio: {n} clips, {need:.2f} GB as int16, kept on {'the GPU' if dev == 'cuda' else 'the CPU'} · {time.time() - t0:.0f} s", flush=True)


def fetch(ids):
    # clips `ids` of the training lists → float32 waves in [−1, 1] and their classes, on the device
    j = torch.from_numpy(np.ascontiguousarray(ids)).to(AUDIO.device)
    return AUDIO[j].to(DEVICE).float() / 32768.0, LABELS[j].to(DEVICE)


def build_net(seed):
    torch.manual_seed(seed)
    return TASTgramMFN(len(CLASSES), N_FRAMES, LAST_KERNEL, c_dim=SV["c_dim"], win_len=SV["win_len"], hop_len=SV["hop"], m=SV["arc_m"], s=SV["arc_s"])


def mixed_step(net, x, y, lam, perm):
    # the reference's mixup_data and noisy_arcmix_criterion: the waves and the log-mels are mixed with one λ and one permutation;
    # the margin is at the first clip's class; the loss is λ·CE(first class) + (1 − λ)·CE(second class) on the same logits
    mel = log_mel(x)
    logits, _ = net((lam * x + (1 - lam) * x[perm]).unsqueeze(1), lam * mel + (1 - lam) * mel[perm], y)
    return lam * CE(logits, y) + (1 - lam) * CE(logits, y[perm]), logits


def pick_memory_format():
    # the same mathematics in two memory layouts: a few training steps are timed in each on this GPU, and the faster one is kept.
    # These are also the first real training steps: if neither layout runs with the audio on the GPU (most likely the GPU's memory
    # is too small for both), the audio is moved to the CPU's memory and the steps are tried once more.
    global MEMFMT, AUDIO, LABELS, STORE_DEV
    if DEVICE != "cuda":
        return
    err = ""
    for attempt in (0, 1):
        MF_TIMES.clear()
        x, y = fetch(np.arange(min(SV["batch"], len(ITEMS)))); perm = torch.arange(len(y) - 1, -1, -1, device=DEVICE)
        for name, fmt in (("contiguous", torch.contiguous_format), ("channels_last", torch.channels_last)):
            MEMFMT = fmt; net = opt = scaler = loss = None
            try:
                net = build_net(0).to(DEVICE).to(memory_format=fmt); net.train()
                opt = torch.optim.AdamW(net.parameters(), lr=SV["lr"]); scaler = torch.amp.GradScaler("cuda", enabled=AMP)
                for i in range(16):
                    if i == 4:
                        torch.cuda.synchronize(); t0 = time.time()
                    loss, _ = mixed_step(net, x, y, 0.5, perm)
                    opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
                torch.cuda.synchronize(); MF_TIMES[name] = round((time.time() - t0) / 12, 4)
            except Exception as e:
                MF_TIMES[name] = None; err = repr(e)[:160]
                print(f"  ({name} could not be timed: {err})", flush=True)
            del net, opt, scaler, loss
            gc.collect(); torch.cuda.empty_cache()
        ok = {k: v for k, v in MF_TIMES.items() if v}
        del x, y, perm
        if ok or STORE_DEV != "cuda":
            break
        AUDIO, LABELS, STORE_DEV = AUDIO.cpu(), LABELS.cpu(), "cpu"
        gc.collect(); torch.cuda.empty_cache()
        print("  no training step ran with the audio on the GPU: the audio is moved to the CPU's memory, and the steps are tried again", flush=True)
    if not ok:
        raise SystemExit(f"stop: a training step does not run on this GPU ({err}). Choose G4 in Runtime › Change runtime type, then Run all.")
    best = min(ok, key=ok.get)
    MEMFMT = torch.channels_last if best == "channels_last" else torch.contiguous_format
    print(f"memory layout: seconds per training step {MF_TIMES} → {best}", flush=True)


@torch.no_grad()
def hold_out_pass(net, hold):
    # as the reference's valid(): evaluation mode, margin at the clip's class, no mixing; the mean of the batches' means
    net.eval(); losses, accs = [], []
    for i in range(0, len(hold), SV["batch"]):
        x, y = fetch(hold[i:i + SV["batch"]])
        logits, _ = net(x.unsqueeze(1), log_mel(x), y)
        losses.append(float(CE(logits, y))); accs.append(float((logits.argmax(1) == y).float().mean()))
    return float(np.mean(losses)), float(np.mean(accs))


def fmt_time(s):
    return f"{s / 3600:.1f} h" if s >= 5400 else f"{s / 60:.0f} min"


def hold_out_split():
    # 5 % held out at random, one draw for the three seeds (as notebook 25): the held-out clips are then unseen by every seed's model.
    # The draw is saved in Drive the first time and read from there afterwards, so that a later session uses the same clips.
    n = len(ITEMS); n_tr = int(n * SV["split_ratio"]); f = OUT / "hold_out.json"
    if f.exists():
        d = json.loads(f.read_text())
        assert d["n_clips"] == n and len(d["hold_out"]) == n - n_tr, "the saved hold-out belongs to other training lists"
        hold = np.array(d["hold_out"], dtype=np.int64)
    else:
        hold = np.sort(np.random.default_rng([30, 0]).permutation(n)[n_tr:])
        f.write_text(json.dumps({"n_clips": int(n), "hold_out": hold.tolist()}))
    return np.setdiff1d(np.arange(n), hold), hold


def train_sv(seed):
    # one training of the second view: 5 % held out, Noisy-ArcMix on every batch, AdamW with cosine annealing per epoch, the model
    # with the lowest hold-out loss kept (as the reference); saved to Drive every SV["save_every"] epochs and resumed from there
    fin, res = OUT / f"sv_s{seed}.pt", OUT / f"sv_s{seed}_resume.pt"
    build_audio()
    tr_idx, hold = hold_out_split(); n, n_tr = len(ITEMS), len(tr_idx)
    hold_md5 = hashlib.md5(hold.astype("<i8").tobytes()).hexdigest()
    bs, epochs = SV["batch"], SV["epochs"]; steps = math.ceil(n_tr / bs)
    net = build_net(seed).to(DEVICE).to(memory_format=MEMFMT)
    opt = torch.optim.AdamW(net.parameters(), lr=SV["lr"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=SV["eta_min_ratio"] * SV["lr"])
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    log = {"seed": seed, "cfg": dict(SV), "classes": CLASSES, "n_clips": int(n), "n_train": int(n_tr), "n_hold_out": int(n - n_tr),
           "hold_out_md5": hold_md5, "steps_per_epoch": int(steps), "n_frames": int(N_FRAMES), "last_kernel": list(LAST_KERNEL),
           "parameters": int(sum(p.numel() for p in net.parameters())), "epochs": [], "resumed_at_epoch": [], "sessions": []}
    best, start = {"val_loss": float("inf"), "epoch": -1, "state": None}, 0
    if res.exists():
        st = torch.load(res, map_location="cpu", weights_only=True)
        log = json.loads(st["log"])
        assert log["classes"] == CLASSES and log["hold_out_md5"] == hold_md5 and log["cfg"] == json.loads(json.dumps(SV)), \
            "the saved training used other lists or settings"
        net.load_state_dict(st["model"]); opt.load_state_dict(st["opt"]); sched.load_state_dict(st["sched"]); scaler.load_state_dict(st["scaler"])
        best = {"val_loss": st["best_val_loss"], "epoch": st["best_epoch"], "state": st["best_state"]}
        start = st["epoch"] + 1; log["resumed_at_epoch"].append(start)
        print(f"seed {seed}: resuming after epoch {start}", flush=True)
    session = {"gpu": GPU, "memory_layout": "channels_last" if MEMFMT == torch.channels_last else "contiguous", "audio_on": STORE_DEV,
               "from_epoch": start, "step_seconds": None}
    log["sessions"].append(session)
    print(f"seed {seed}: {n_tr} training clips and {n - n_tr} held out · {len(CLASSES)} classes · {steps} steps per epoch · "
          f"{log['parameters']:,} parameters", flush=True)
    t_from, t_to = min(5, steps - 2), min(45, steps - 1)
    for ep in range(start, epochs):
        rng = np.random.default_rng([seed, 30, 1, ep]); order = tr_idx[rng.permutation(n_tr)]
        lr_now = opt.param_groups[0]["lr"]
        net.train(); t0, tot, acc = time.time(), 0.0, 0.0
        for b in range(steps):
            x, y = fetch(order[b * bs:(b + 1) * bs])
            lam = float(rng.beta(SV["alpha"], SV["alpha"])); pb = torch.from_numpy(rng.permutation(len(y))).to(DEVICE)
            loss, logits = mixed_step(net, x, y, lam, pb)
            if not bool(torch.isfinite(loss)):
                raise SystemExit(f"stop: G1 failed. The training loss of seed {seed} is not finite (epoch {ep + 1}, step {b + 1})")
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
            tot += float(loss.detach()); acc += float((logits.detach().argmax(1) == y).float().mean())
            if ep == start and b in (t_from, t_to) and t_to > t_from >= 0:
                if DEVICE == "cuda":
                    torch.cuda.synchronize()
                if b == t_from:
                    t_mark = time.time()
                else:
                    session["step_seconds"] = round((time.time() - t_mark) / (t_to - t_from), 4)
                    left_here = (epochs - ep) * steps * session["step_seconds"] * 1.03
                    later = [s for s in SV["seeds"] if s != seed and not (OUT / f"sv_s{s}.pt").exists()]
                    left_all = left_here + len(later) * epochs * steps * session["step_seconds"] * 1.03
                    print(f"  this GPU ({GPU or 'CPU'}): {session['step_seconds']:.3f} s per step → about {fmt_time(left_here)} left for this "
                          f"training, about {fmt_time(left_all)} for all the trainings that are left", flush=True)
                    if left_all > 6 * 3600 and not TEST:
                        print("  NOTE: that is a long run on this GPU. It can stay as it is (it resumes after a disconnect). For a much shorter "
                              "run: Runtime › Change runtime type › G4, then Runtime › Run all; what is saved in Drive is kept.", flush=True)
        sched.step()
        vl, va = hold_out_pass(net, hold)
        if not math.isfinite(vl):
            raise SystemExit(f"stop: G1 failed. The hold-out loss of seed {seed} is not finite (epoch {ep + 1})")
        if best["val_loss"] > vl:                                     # the reference saves the model whenever the hold-out loss falls
            best = {"val_loss": vl, "epoch": ep, "state": {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}}
        log["epochs"].append({"epoch": ep, "train_loss": round(tot / steps, 5), "train_accuracy": round(acc / steps, 5), "hold_out_loss": round(vl, 5),
                              "hold_out_accuracy": round(va, 5), "lr": lr_now, "seconds": round(time.time() - t0, 1)})
        if ep < start + 3 or (ep + 1) % 5 == 0 or ep + 1 == epochs:
            print(f"seed {seed} · epoch {ep + 1}/{epochs} · loss {tot / steps:.4f} · hold-out loss {vl:.4f}, accuracy {va:.4f} · "
                  f"best epoch {best['epoch'] + 1} · {time.time() - t0:.0f} s", flush=True)
        if (ep + 1) % SV["save_every"] == 0 and ep + 1 < epochs:
            tmp = res.with_name(res.name + ".tmp")
            torch.save({"model": {k: v.detach().cpu() for k, v in net.state_dict().items()}, "opt": opt.state_dict(), "sched": sched.state_dict(),
                        "scaler": scaler.state_dict(), "epoch": ep, "best_state": best["state"], "best_epoch": best["epoch"],
                        "best_val_loss": best["val_loss"], "log": json.dumps(log)}, tmp)
            os.replace(tmp, res)
    log["best_epoch"], log["best_hold_out_loss"] = int(best["epoch"]), float(best["val_loss"])
    tmp = fin.with_name(fin.name + ".tmp")
    torch.save({"state": best["state"], "last_state": {k: v.detach().cpu() for k, v in net.state_dict().items()}, "best_epoch": int(best["epoch"]),
                "best_val_loss": float(best["val_loss"]), "classes": CLASSES, "hold_out": hold.tolist(), "log": json.dumps(log)}, tmp)
    os.replace(tmp, fin)
    res.unlink(missing_ok=True)
    del net, opt, sched, scaler
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


@torch.no_grad()
def embed_sv(seed):
    # the saved model on every DCASE clip: the 128-d feature, and the paper's score (cross-entropy for the clip's own machine class)
    fin = OUT / f"sv_s{seed}.pt"
    st = torch.load(fin, map_location="cpu", weights_only=True)
    assert st["classes"] == CLASSES, "the saved model used other classes"
    net = build_net(seed); net.load_state_dict(st["state"]); net.to(DEVICE).to(memory_format=MEMFMT).eval()
    out, bs = {}, SV["embed_batch"]
    for t in TYPES:
        y_all = np.array([CLS_OF[(t, q)] for q in DMETA[t]["mid"]], dtype=np.int64); n = DST[t].shape[0]
        V, ce = np.zeros((n, SV["c_dim"]), np.float32), np.zeros(n, np.float32)
        for i in range(0, n, bs):
            x = torch.from_numpy(f32(DST[t][i:i + bs])).to(DEVICE); y = torch.from_numpy(y_all[i:i + bs]).to(DEVICE)
            logits, feat = net(x.unsqueeze(1), log_mel(x), y)
            V[i:i + len(y)] = feat.cpu().numpy(); ce[i:i + len(y)] = F.cross_entropy(logits, y, reduction="none").cpu().numpy()
        assert np.isfinite(V).all() and np.isfinite(ce).all(), f"non-finite features for {t}"
        out[f"V__{t}"], out[f"CE__{t}"] = V, ce
    tmp = OUT / f"emb_SV_s{seed}.part.npz"
    np.savez(tmp, **out, meta=json.dumps({"seed": seed, "weights_md5": md5(fin), "best_epoch": int(st["best_epoch"]), "test_run": TEST,
                                          "date": time.strftime("%Y-%m-%d %H:%M")}))
    os.replace(tmp, OUT / f"emb_SV_s{seed}.npz")
    del net
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

In [ ]:
# 13 · How systems are scored on a section's test clips, and the official evaluator's metrics: per section AUC (source), AUC (target)
# and pAUC; per type and overall their harmonic means
VIEWS, CES, VM = {}, {}, {}                                           # seed → type → features (n, 128) and the paper's score (n,)
HOLD_ROWS, CAL_P = {}, {}                                             # type → store rows of the held-out clips; (seeds, type) → the view's spread
SYSTEMS = ["ZP", "frozen", "view_paper", "ZP_AT", "frozen_AT", "view_LDN1", "view_B1", "frozen_B", "frozen_B3", "echo"]
LABEL = {"ZP": "as it is · frozen + view (ZP)", "frozen": "as it is · frozen recipe alone", "view_paper": "view alone · paper's score",
         "ZP_AT": "AT · frozen (attributes) + view", "frozen_AT": "AT · frozen recipe alone",
         "view_LDN1": "view alone · LDN1", "view_B1": "view alone · B1", "frozen_B": "frozen · BEATs iter3+ part",
         "frozen_B3": "frozen · BEATs iter3 part", "echo": "ECHO alone · LDN1"}
VALUES = ("AUC_source", "AUC_target", "pAUC")                         # a section's three values, as the official evaluator names them


def view_matrix(seeds, t):
    # the program's seed ensemble: each seed's features unit-normalised, concatenated, ÷√seeds
    k = (tuple(seeds), t)
    if k not in VM:
        VM[k] = np.concatenate([unit(VIEWS[s][t]) for s in seeds], 1) / math.sqrt(len(seeds))
    return VM[k]


def calibrate_paper(seeds, t):
    # form ZP's calibration of the view: the log of the paper's score (averaged over the seeds) on the held-out clips. No test clip.
    k = (tuple(seeds), t)
    if k not in CAL_P:
        rows = np.asarray(HOLD_ROWS[t])
        CAL_P[k] = spread(np.log(np.maximum(sum(CES[s][t][rows] for s in seeds) / len(seeds), LDN_FLOOR)) if len(rows) else np.zeros(0))
    return CAL_P[k]


def paper(seeds, t, rows):
    # the paper's score (the cross-entropy for the clip's own section class, margin on), averaged over the seeds
    return sum(CES[s][t][rows] for s in seeds) / len(seeds)


def zp_ok(seeds):
    # form ZP needs a spread from at least CAL_MIN clips, for both systems and every type
    return all(c["n"] >= CAL_MIN and c["sigma"] is not None and c["sigma"] > 1e-9 for t in TYPES for c in (CAL_F[t], CAL_F_AT[t], calibrate_paper(seeds, t)))


def score_of(name, t, set_, q, seeds):
    # one system's anomaly scores of one section's test clips; the references are all of the section's normal training clips
    ref, te = dc_rows(t, set_, "train", q), dc_rows(t, set_, "test", q)
    if name == "frozen":
        return frozen_of(t, set_, q)
    if name == "view_paper":
        return paper(seeds, t, te)
    if name == "frozen_AT":                                           # the frozen recipe trained with attribute classes (computed in cell 11)
        return SF_AT[(t, set_, q)]
    if name == "ZP_AT":
        return (np.log(np.maximum(SF_AT[(t, set_, q)], LDN_FLOOR)) / CAL_F_AT[t]["sigma"]
                + np.log(np.maximum(paper(seeds, t, te), LDN_FLOOR)) / calibrate_paper(seeds, t)["sigma"])
    if name == "ZP":                                                  # each system's log score in units of its spread on unseen normal clips
        return (np.log(np.maximum(frozen_of(t, set_, q), LDN_FLOOR)) / CAL_F[t]["sigma"]
                + np.log(np.maximum(paper(seeds, t, te), LDN_FLOOR)) / calibrate_paper(seeds, t)["sigma"])
    if name in ("frozen_B", "frozen_B3"):
        return part_score(name.split("_")[1], t, ref, te)
    if name == "echo":
        return echo_score(t, ref, te)
    V = view_matrix(seeds, t)
    if name == "view_LDN1":
        return ldn1(V[ref], V[te])
    if name == "view_B1":
        return b1(V[ref], V[te])
    raise ValueError(name)


def hm(v):
    # the harmonic mean, as the official evaluator takes it
    v = np.maximum(np.asarray(v, dtype=np.float64), sys.float_info.epsilon)
    return float(len(v) / np.sum(1.0 / v))


def section_metrics(y, s, dom):
    # the official evaluator's values of one section. AUC of a domain: that domain's normal clips against all the section's anomalous
    # clips, whatever their domain. pAUC: all the section's clips, max_fpr = 0.1
    y, s, dom = np.asarray(y), np.asarray(s, dtype=np.float64), np.asarray(dom)
    out = {"n_test": int(len(y)), "n_anomalies": int(y.sum())}
    for d in ("source", "target"):
        k = (dom == d) | (y != 0)
        out[f"AUC_{d}"] = float(roc_auc_score(y[k], s[k])); out[f"n_normal_{d}"] = int(((dom == d) & (y == 0)).sum())
    out["pAUC"] = float(roc_auc_score(y, s, max_fpr=0.1)); out["AUC_all"] = float(roc_auc_score(y, s))
    return out


def summary(per):
    # per: type → section → its values. The type's score is the harmonic mean of its sections' three values; the official score Ω is the
    # harmonic mean of every section's three values; the source, target and pAUC lines are harmonic means over the sections
    types = {t: {"score": hm([d[q][v] for q in d for v in VALUES]), **{v: hm([d[q][v] for q in d]) for v in VALUES}, "per_section": d} for t, d in per.items()}
    return {"omega": hm([d[q][v] for d in per.values() for q in d for v in VALUES]), **{v: hm([d[q][v] for d in per.values() for q in d]) for v in VALUES},
            "values": int(sum(len(d) * len(VALUES) for d in per.values())), "per_type": types}


def dev_eval(seeds, names, keep=None):
    # the development set under each system. Its labels and domains are in the file names. With `keep`, every clip's scores are kept too
    names = [nm for nm in names if nm not in ("ZP", "ZP_AT") or zp_ok(seeds)]   # the calibrated form only when its spreads can be formed
    out = {}
    for nm in names:
        per = {}
        for t in TYPES:
            m = DMETA[t]; lab, dom, fn = np.array(m["label"]), np.array(m["domain"]), np.array(m["name"])
            for q in machines(t, "dev"):
                te = dc_rows(t, "dev", "test", q); sc = score_of(nm, t, "dev", q, seeds)
                per.setdefault(t, {})[q] = section_metrics(lab[te], sc, dom[te])
                if keep is not None:
                    keep.setdefault((t, q), {"file": fn[te].tolist(), "label": lab[te].tolist(), "domain": dom[te].tolist()})[nm] = np.asarray(sc, dtype=np.float64)
        out[nm] = summary(per)                                        # the development types only: the others have no labelled clip
    FM.clear(); VM.clear(); gc.collect()
    return out


def fusion_word(kept, frozen, view):
    # the rules' three words for the fusion, on official scores of four decimals: against the better of its two parts
    d = round(round(kept, 4) - max(round(frozen, 4), round(view, 4)), 4)
    return ("helps" if d >= Q3_BAR else "costs" if d < -Q3_BAR else "no clear difference"), d


def fusion_table(res):
    # the kept system's official score less the frozen recipe's and less the view's, and the word
    k, f, v = (res[nm]["omega"] for nm in ("ZP", "frozen", "view_paper")); w, d = fusion_word(k, f, v)
    return {"kept": round(k, 4), "frozen": round(f, 4), "view": round(v, 4), "kept_less_frozen": round(round(k, 4) - round(f, 4), 4),
            "kept_less_view": round(round(k, 4) - round(v, 4), 4), "kept_less_better_part": d, "word": w}


def shift_table(res):
    # question 2: per system, the harmonic means over the sections of AUC (source) and AUC (target), and the difference
    return {nm: {"AUC_source": round(res[nm]["AUC_source"], 4), "AUC_target": round(res[nm]["AUC_target"], 4),
                 "source_less_target": round(round(res[nm]["AUC_source"], 4) - round(res[nm]["AUC_target"], 4), 4)} for nm in res}


assert [fusion_word(0.65, 0.64, 0.60)[0], fusion_word(0.6499, 0.64, 0.60)[0], fusion_word(0.63, 0.64, 0.60)[0], fusion_word(0.6299, 0.64, 0.60)[0]] == \
    ["helps", "no clear difference", "no clear difference", "costs"], "stop: the fusion's words are not the rules'"
# the metrics on a hand-made section: four normal clips (two per domain) and two anomalous ones
_m = section_metrics([0, 0, 0, 0, 1, 1], [0.1, 0.4, 0.3, 0.9, 0.5, 0.8], ["source", "source", "target", "target", "source", "target"])
assert (_m["AUC_source"], _m["AUC_target"], _m["AUC_all"]) == (1.0, 0.5, 0.75) and abs(hm([0.5, 1.0]) - 2 / 3) < 1e-12, "stop: the metrics are not the evaluator's"


def show(res, names, title):
    print(title)
    print(f"  {'':<36}{'Ω':>9}{'source':>9}{'target':>9}{'pAUC':>9}   " + "".join(f"{NAME[t][:10]:>11}" for t in next(iter(res.values()))["per_type"]))
    for nm in [n_ for n_ in names if n_ in res]:
        r = res[nm]
        print(f"  {LABEL[nm]:<36}{r['omega']:>9.4f}{r['AUC_source']:>9.4f}{r['AUC_target']:>9.4f}{r['pAUC']:>9.4f}   " + "".join(f"{v_['score']:>11.4f}" for v_ in r["per_type"].values()))

In [ ]:
# 14 · Part 2 · The view's three seeds (each: training or resume, then its features of every clip); gates G0(d) and G1. No test clip
# is scored in this cell: the development set is read only after every training, as the rules order it.
t_p2 = time.time()
ITEMS, CLASSES, CLS_OF = training_lists()
N_FRAMES = frames_of(NS); LAST_KERNEL = last_kernel(SV["n_mels"], N_FRAMES)
_, HOLD = hold_out_split(); N_TRAIN = len(ITEMS) - len(HOLD)
HOLD_ROWS.update({t: ITEMS[HOLD][ITEMS[HOLD][:, 0] == ti][:, 1] for ti, t in enumerate(TYPES)})
print(f"the view's training lists: {len(ITEMS)} clips ({int(sum(1 for c in CLASSES if c.endswith('additional')))} evaluation types among "
      f"{len(CLASSES)} classes) · {N_FRAMES} frames · last kernel {LAST_KERNEL} · {N_TRAIN} train, {len(HOLD)} held out (the same clips for every seed)")
assert CLASSES == F_CLASSES, "stop: the view's classes are not the frozen recipe's types"
G0["d"] = {"ok": bool((N_FRAMES, LAST_KERNEL, len(CLASSES)) == (REF_FRAMES, REF_KERNEL, REF_CLASSES)), "frames": int(N_FRAMES),
           "last_kernel": list(LAST_KERNEL), "classes": len(CLASSES), "clips": int(len(ITEMS)), "train": int(N_TRAIN), "held_out": int(len(HOLD)),
           "held_out_per_type": {t: int(len(HOLD_ROWS[t])) for t in TYPES}}
print(f"G0(d) · the view has the reference's sizes (313 frames, a last kernel of 8 × 20) and {REF_CLASSES} classes:", G0["d"]["ok"])
if not TEST and not G0["d"]["ok"]:
    raise SystemExit("stop: G0(d) failed, the lists or the sizes are not the rules'")
_todo = [s for s in SV["seeds"] if not (OUT / f"sv_s{s}.pt").exists()]
print(f"trainings to do: {len(_todo)} of {len(SV['seeds'])} · {math.ceil(N_TRAIN / SV['batch'])} steps per epoch, {SV['epochs']} epochs each. "
      f"At the 0.0192 s per step notebook 27 measured on an RTX PRO 6000 (G4), that is about "
      f"{math.ceil(N_TRAIN / SV['batch']) * SV['epochs'] * 0.0192 * 1.05 / 60:.0f} min per training there. The measured estimate for this GPU follows.")
if _todo:
    build_audio(); pick_memory_format()
LOGS, G1, T_TRAIN, T_EMBED = {}, {}, {}, {}


def ensure_sv(seed):
    # one seed: trained (or resumed, or read from Drive), then its features of every DCASE clip; gate G1
    fin, ef = OUT / f"sv_s{seed}.pt", OUT / f"emb_SV_s{seed}.npz"
    if not fin.exists():
        t0 = time.time(); train_sv(seed); T_TRAIN[seed] = round(time.time() - t0, 1)
    if ef.exists() and json.loads(str(np.load(ef, allow_pickle=False)["meta"]))["weights_md5"] != md5(fin):
        ef.unlink()                                                   # features of another model file: computed again
    if not ef.exists():
        t0 = time.time(); embed_sv(seed); T_EMBED[seed] = round(time.time() - t0, 1)
        print(f"seed {seed}: features of every DCASE clip saved · {T_EMBED[seed]:.0f} s", flush=True)
    D = np.load(ef, allow_pickle=False)
    VIEWS[seed] = {t: D[f"V__{t}"].astype(np.float64) for t in TYPES}; CES[seed] = {t: D[f"CE__{t}"].astype(np.float64) for t in TYPES}
    assert all(len(VIEWS[seed][t]) == len(DMETA[t]["name"]) for t in TYPES), "the saved features do not have this session's number of clips"
    st = torch.load(fin, map_location="cpu", weights_only=True)
    assert st["hold_out"] == HOLD.tolist(), "the saved model was trained with another hold-out"
    log = json.loads(st["log"]); LOGS[seed] = log
    e = log["epochs"]
    G1[seed] = {"first_train_loss": e[0]["train_loss"], "last_train_loss": e[-1]["train_loss"], "first_hold_out_loss": e[0]["hold_out_loss"],
                "best_hold_out_loss": round(log["best_hold_out_loss"], 5), "best_epoch": log["best_epoch"] + 1,
                "hold_out_accuracy_at_best": e[log["best_epoch"]]["hold_out_accuracy"], "epochs": len(e),
                "ok": bool(len(e) == SV["epochs"] and e[-1]["train_loss"] < e[0]["train_loss"] and log["best_hold_out_loss"] < e[0]["hold_out_loss"])}
    print(f"G1 · seed {seed}: training loss {e[0]['train_loss']:.4f} → {e[-1]['train_loss']:.4f}; hold-out loss {e[0]['hold_out_loss']:.4f} → "
          f"{log['best_hold_out_loss']:.4f} at epoch {log['best_epoch'] + 1} (accuracy {G1[seed]['hold_out_accuracy_at_best']:.4f}): {G1[seed]['ok']}", flush=True)
    if not G1[seed]["ok"] and not TEST:
        raise SystemExit(f"stop: G1 failed for seed {seed}")


for _seed in SV["seeds"]:
    ensure_sv(_seed)
assert all(LOGS[s]["cfg"] == json.loads(json.dumps(SV)) and LOGS[s]["classes"] == CLASSES for s in SV["seeds"]), "stop: a saved view was trained with other settings"
AUDIO = LABELS = None
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()
T_PART2 = round(time.time() - t_p2, 1)
print(f"the view's trainings are done · {T_PART2 / 60:.0f} min in this cell · {(time.time() - T_START) / 60:.0f} min in this session")

In [ ]:
# 15 · The development set (seven types), read here for the first time. It selects between two systems and nothing else: the system as
# it is, and the one whose frozen recipe was trained with attribute classes (AT). Gate G-DEV, then the choice by the rules.
SEEDS = list(SV["seeds"])
CAL = {"frozen": CAL_F, "frozen_AT": CAL_F_AT, "view_paper": {t: calibrate_paper(SEEDS, t) for t in TYPES}, "frozen_cap": FROZEN_CAP, "minimum_clips": CAL_MIN}
_cs = lambda d: {t: (d[t]["n"], None if d[t]["sigma"] is None else round(d[t]["sigma"], 4)) for t in TYPES}
print("form ZP · the spreads of the log score on normal clips a system did not train on (clips, σ): the frozen recipe", _cs(CAL_F), "· with attribute classes", _cs(CAL_F_AT),
      "· the view by the paper's score", _cs(CAL["view_paper"]))
if not zp_ok(SEEDS):
    raise SystemExit(f"stop: a spread of form ZP cannot be formed (it needs at least {CAL_MIN} clips per type and must not be zero): {CAL}")
DEV_SC = {}                                                           # (type, section) → every development test clip's scores, saved in the zip
DEV = dev_eval(SEEDS, SYSTEMS, keep=DEV_SC)
assert list(DEV) == SYSTEMS and all(DEV[nm]["values"] == 3 * len(TYPES_DEV) and list(DEV[nm]["per_type"]) == TYPES_DEV for nm in SYSTEMS)
show(DEV, SYSTEMS, f"\nDevelopment set · the official score Ω, the harmonic means over the sections of AUC (source), AUC (target) and pAUC, and each "
                   f"type's score; the view with {len(SEEDS)} seeds:")
print("\nDevelopment set · per section, AUC (source) / AUC (target) / pAUC:")
for t in TYPES_DEV:
    for nm in AS_IS + ["ZP_AT", "frozen_AT"]:
        print(f"  {NAME[t]:<11} {LABEL[nm]:<34}" + " · ".join(f"{q[-2:]} {v['AUC_source']:.4f} / {v['AUC_target']:.4f} / {v['pAUC']:.4f}"
                                                               for q, v in DEV[nm]["per_type"][t]["per_section"].items()))
# the baselines' published development tables: the harmonic mean of a baseline's 21 values is its development Ω
PUB = {k: {"omega": hm([x for v in d.values() for x in v]) / 100, **{VALUES[i]: hm([v[i] for v in d.values()]) / 100 for i in range(3)},
           "per_type": {t: hm(v) / 100 for t, v in d.items()}} for k, d in PUBLISHED_DEV.items()}
print("\nPublished development numbers (the task page; yardsticks, not bars):")
for k_, v_ in PUB.items():
    print(f"  {k_:<36}{v_['omega']:>9.4f}{v_['AUC_source']:>9.4f}{v_['AUC_target']:>9.4f}{v_['pAUC']:>9.4f}   " + "".join(f"{v_['per_type'][t]:>11.4f}" for t in v_["per_type"]))
PER_SEED = {s: dev_eval([s], ["ZP", "view_paper", "view_LDN1", "view_B1"]) for s in SEEDS}
print("\nSeed spread · the development Ω with each seed of the view alone:",
      {s: {"kept system": round(PER_SEED[s]["ZP"]["omega"], 4) if "ZP" in PER_SEED[s] else None, "view": round(PER_SEED[s]["view_paper"]["omega"], 4)} for s in SEEDS})
SHIFT_DEV = shift_table({nm: DEV[nm] for nm in AS_IS})
print("\nThe shift on the development set (harmonic mean AUC of the source domain, of the target domain, and the difference; reported):",
      {LABEL[nm]: (v["AUC_source"], v["AUC_target"], v["source_less_target"]) for nm, v in SHIFT_DEV.items()})
FUSION_DEV = fusion_table(DEV)
print("The fusion on the development set (the kept system's Ω less the better of its two parts; reported, no decision):",
      (FUSION_DEV["kept_less_better_part"], FUSION_DEV["word"]))

# gate G-DEV, before the evaluation set is touched: it is there to catch a broken pipeline, not to judge the system. It reads the source
# domain, the conditions a system trained on: under a change of conditions a working system's AUC (target) can fall below 0.50
GDEV = {nm: {"AUC_source": round(DEV[nm]["AUC_source"], 4), "bar": GDEV_BAR, "ok": bool(round(DEV[nm]["AUC_source"], 4) >= GDEV_BAR), "gates": nm in GDEV_SYSTEMS}
        for nm in AS_IS}
GDEV_OK = bool(all(v["ok"] for v in GDEV.values() if v["gates"]))
print(f"\nG-DEV · the system as it is has a source-domain line (harmonic mean AUC of the source domain) of at least {GDEV_BAR} on the development set:", GDEV_OK,
      "· its two parts are reported, and do not stop the run:", GDEV)
if not GDEV_OK:
    raise SystemExit("stop: G-DEV failed; something is broken, and the evaluation set is not scored")
# the choice, by the rules: AT is the result if its development official score is at least ADOPT_BAR above the system as it is, both
# rounded to four decimals. Otherwise the system as it is is the result.
CHOICE = {"as_it_is": round(DEV["ZP"]["omega"], 4), "attribute_classes": round(DEV["ZP_AT"]["omega"], 4), "bar": ADOPT_BAR,
          "frozen_as_it_is": round(DEV["frozen"]["omega"], 4), "frozen_attribute_classes": round(DEV["frozen_AT"]["omega"], 4)}
CHOICE["attribute_classes_less_as_it_is"] = round(CHOICE["attribute_classes"] - CHOICE["as_it_is"], 4)
CHOICE["adopted"] = bool(CHOICE["attribute_classes_less_as_it_is"] >= ADOPT_BAR)
RESULT_IS = "kept_AT" if CHOICE["adopted"] else "kept"
print(f"\nThe choice · development official score: as it is {CHOICE['as_it_is']:.4f}, with attribute classes {CHOICE['attribute_classes']:.4f}; the difference is "
      f"{CHOICE['attribute_classes_less_as_it_is']:+.4f} (attribute classes are the result from {ADOPT_BAR:+.2f}) → THE RESULT IS",
      "THE SYSTEM WITH ATTRIBUTE CLASSES" if CHOICE["adopted"] else "THE SYSTEM AS IT IS")

In [ ]:
# 16 · The evaluation set (eight types), scored once. Five sets of score files are written before any label is read: the system as it
# is, the system with attribute classes, the frozen recipe of each, and the view. Which of the first two is the result was fixed by the
# development set in cell 15. The ground truth is fetched only after that. A later run reads the saved results and scores nothing again.
EV_DIR = OUT / "dcase2025_evaluation"
EV_RES, EV_LOCK = EV_DIR / "results.json", EV_DIR / "scores_written.json"
EV_SETS = {"kept": "ZP", "frozen": "frozen", "view": "view_paper", "kept_AT": "ZP_AT", "frozen_AT": "frozen_AT"}    # folder → system
EV_DIR.mkdir(exist_ok=True)
if EV_LOCK.exists() and json.loads(EV_LOCK.read_text())["result_is"] != RESULT_IS:
    raise SystemExit("stop: the evaluation set was already scored with another choice of the result; the development set's choice must not change")


def fetch_evaluator():
    # the official evaluator's repository at the pinned commit: the script, its two tool files and the ground-truth files of the
    # evaluation set, checked against the recorded hashes before anything is used
    d = WORK / "evaluator2025"
    if not (d / "checked").exists():
        z = WORK / EV_ARCHIVE[1]
        download(EV_ARCHIVE[0], z)
        with zipfile.ZipFile(z) as zf:
            names = {n.split("/", 1)[1]: n for n in zf.namelist() if "/" in n and not n.endswith("/")}
            want = sorted(n for n in names if n.startswith(tuple(p + "/" for p in GT_PARTS)) and n.endswith(".csv"))
            code = [EV_SCRIPT] + EV_TOOLS
            if any(n not in names for n in code) or not want:
                raise SystemExit("stop: the evaluator's repository does not hold the script, its tools and the ground truth")
            blobs = {n: zf.read(names[n]) for n in code + want}
        h, hc = hashlib.sha256(), hashlib.sha256()
        for n in want:                                                # in path order
            h.update(n.encode() + b"\0" + blobs[n])
        for n in code:
            hc.update(n.encode() + b"\0" + blobs[n])
        if hc.hexdigest() != EV_SHA or h.hexdigest() != GT_SHA:
            raise SystemExit("stop: the evaluator's repository is not the recorded one (the script or a ground-truth file differs)")
        shutil.rmtree(d, ignore_errors=True)
        for n, b in blobs.items():
            (d / n).parent.mkdir(parents=True, exist_ok=True); (d / n).write_bytes(b)
        (d / "checked").write_text(f"{len(want)} ground-truth files")
    return d


def read_ground_truth():
    # the labels and the domains of the evaluation test clips: (type, section) → file name → (label, domain). The repository names a
    # clip as the evaluation zip does (section_00_NNNN.wav)
    d = fetch_evaluator(); gt = {}
    for t in TYPES_EVAL:
        for q in EVAL_IDS[t]:
            with open(d / GT_PARTS[0] / f"ground_truth_{t}_{q}_test.csv", newline="") as fh:
                lab = [r for r in csv.reader(fh) if r]
            with open(d / GT_PARTS[1] / f"ground_truth_{t}_{q}_test.csv", newline="") as fh:
                dom = [r for r in csv.reader(fh) if r]
            assert [r[0] for r in lab] == [r[0] for r in dom] and len(set(r[0] for r in lab)) == len(lab), (t, q)
            gt[(t, q)] = {a[0]: (int(float(a[1])), "target" if int(float(b[1])) == 1 else "source") for a, b in zip(lab, dom)}
    return d, gt


def read_result(path):
    # one result file of the official evaluator: per type and section its AUC (source), AUC (target) and pAUC, and the lines over all
    res, cur = {"per_section": {}}, None
    with open(path, newline="") as fh:
        for row in csv.reader(fh):
            if len(row) == 1 and row[0]:
                cur = row[0]
            elif len(row) >= 5 and re.fullmatch(r"\d\d", row[0]) and cur:
                res["per_section"].setdefault(cur, {})["section_" + row[0]] = {"AUC_source": float(row[2]), "AUC_target": float(row[3]), "pAUC": float(row[4]), "AUC_all": float(row[1])}
            elif row and row[0] == "official score":
                res["omega"] = float(row[2])
            elif row and row[0].startswith("source harmonic mean over all"):
                res["AUC_source"] = float(row[2])
            elif row and row[0].startswith("target harmonic mean over all"):
                res["AUC_target"] = float(row[2])
    return res


def official_check():
    # the official evaluator (the pinned file, unmodified) on the three sets of score files, compared with this notebook's numbers
    out = {k: {"ran": False} for k in EV_SETS}
    info = {"archive": EV_ARCHIVE[0], "commit": EV_COMMIT, "script_sha256": EV_SHA, "ground_truth_sha256": GT_SHA}

    def stop(reason):
        return {k: {"ran": False, "reason": reason, "evaluator": info} for k in EV_SETS}
    lock = json.loads(EV_LOCK.read_text())
    if not EV_RES.exists() or any(not (EV_DIR / k / f).exists() for k in EV_SETS for f in lock["files"][k] + lock["decision_files"][k]):
        return stop("a score file or results.json of this notebook is missing")
    try:
        d = fetch_evaluator()
    except (SystemExit, Exception) as e:
        return stop(f"the evaluator's repository did not download, or it is not the recorded one ({str(e)[:160]})")
    wd = WORK / "official_evaluator_2025"
    shutil.rmtree(wd, ignore_errors=True); wd.mkdir(parents=True)
    for p in GT_PARTS:
        shutil.copytree(d / p, wd / p)
    for n in [EV_SCRIPT] + EV_TOOLS:
        (wd / n).parent.mkdir(parents=True, exist_ok=True); shutil.copy(d / n, wd / n)
    for k in EV_SETS:                                                 # the evaluator reads ./teams/<team>/<system>/
        (wd / "teams" / "P04" / k).mkdir(parents=True)
        for f in lock["files"][k] + lock["decision_files"][k]:
            shutil.copy(EV_DIR / k / f, wd / "teams" / "P04" / k / f)
    r = subprocess.run([sys.executable, EV_SCRIPT], cwd=wd, capture_output=True, text=True)   # the official script, unmodified
    if r.returncode != 0:
        return stop(f"the evaluator stopped (code {r.returncode}): {(r.stderr or r.stdout)[-300:]}")
    ours = json.loads(EV_RES.read_text())["systems"]
    for k in EV_SETS:
        res_f = wd / "teams_result" / f"{k}_result.csv"
        if not res_f.exists():
            out[k] = {"ran": False, "reason": "the evaluator wrote no result file for this set", "evaluator": info}
            continue
        off = read_result(res_f); mine = ours[k]
        if set(off["per_section"]) != set(TYPES_EVAL) or any(set(off["per_section"][t]) != set(mine["per_type"][t]["per_section"]) for t in TYPES_EVAL) or "omega" not in off:
            out[k] = {"ran": False, "reason": "the evaluator's result file does not list the same machine types and sections", "evaluator": info}
            continue
        diffs = [abs(off["per_section"][t][q][v] - mine["per_type"][t]["per_section"][q][v]) for t in TYPES_EVAL for q in off["per_section"][t] for v in VALUES + ("AUC_all",)]
        diffs += [abs(off["omega"] - mine["omega"]), abs(off["AUC_source"] - mine["AUC_source"]), abs(off["AUC_target"] - mine["AUC_target"])]
        out[k] = {"ran": True, "agree": bool(max(diffs) <= EVAL_TOL), "max_abs_difference": float(max(diffs)), "values_compared": len(diffs), "evaluator": info,
                  "omega": off["omega"], "AUC_source": off["AUC_source"], "AUC_target": off["AUC_target"], "per_section": off["per_section"], "this_notebook_omega": mine["omega"]}
        shutil.copy(res_f, EV_DIR / k / "official_evaluator_result.csv")
    return out


if EV_RES.exists():
    EVAL = json.loads(EV_RES.read_text())
    print(f"the evaluation set was scored on {EVAL['scores_written']}; its saved results are read, and nothing is scored again")
else:
    if not EV_LOCK.exists():
        files, decisions = {k: [] for k in EV_SETS}, {k: [] for k in EV_SETS}
        for k in EV_SETS:
            (EV_DIR / k).mkdir(exist_ok=True)
        for t in TYPES:
            for q in machines(t, "eval"):
                te = dc_rows(t, "eval", "test", q); names = [DMETA[t]["name"][i] for i in te]
                for k, nm in EV_SETS.items():
                    sc = score_of(nm, t, "eval", q, SEEDS)
                    assert len(sc) == len(te) and np.isfinite(sc).all(), (k, t, q)
                    f, g = EV_DIR / k / f"anomaly_score_{t}_{q}_test.csv", EV_DIR / k / f"decision_result_{t}_{q}_test.csv"
                    f.write_text("".join(f"{a},{float(b)!r}\n" for a, b in sorted(zip(names, sc))))
                    g.write_text("".join(f"{a},0\n" for a in sorted(names)))  # the evaluator needs a decision file; no threshold is set in this stage
                    files[k].append(f.name); decisions[k].append(g.name)
            FM.clear(); VM.clear()
        EV_LOCK.write_text(json.dumps({"written": time.strftime("%Y-%m-%d %H:%M"), "files": files, "decision_files": decisions, "test_run": TEST, "result_is": RESULT_IS,
                                       "choice": CHOICE, "systems": EV_SETS, "view_seeds": SEEDS, "frozen": [[e, sd] for e, _, sd in FROZEN], "calibration": CAL}))
        print(f"the evaluation scores of the two systems and of their parts are written ({sum(len(v) for v in files.values())} files), "
              "before any label is read", flush=True)
    lock = json.loads(EV_LOCK.read_text())
    _d, GT = read_ground_truth()                                      # the labels and the domains: fetched and read only here
    for p in GT_PARTS:                                                # the ground-truth rows, kept beside the scores
        (EV_DIR / p).mkdir(exist_ok=True)
        for t in TYPES_EVAL:
            for q in EVAL_IDS[t]:
                shutil.copy(_d / p / f"ground_truth_{t}_{q}_test.csv", EV_DIR / p / f"ground_truth_{t}_{q}_test.csv")
    _sys = {}
    for k in EV_SETS:
        per = {}
        for fn in lock["files"][k]:
            t, q = re.match(r"anomaly_score_(\w+?)_(section_\d+)_test\.csv$", fn).groups()
            rows = [ln.split(",") for ln in (EV_DIR / k / fn).read_text().splitlines() if ln]
            num = [a for a, _ in rows]                                # joined to the ground truth by the file name
            assert sorted(num) == sorted(GT[(t, q)]), f"{fn}: the clips are not the ground truth's"
            per.setdefault(t, {})[q] = section_metrics([GT[(t, q)][n][0] for n in num], [float(b) for _, b in rows], [GT[(t, q)][n][1] for n in num])
        _sys[k] = {"system": EV_SETS[k], **summary({t: per[t] for t in TYPES_EVAL})}
    EVAL = {"scored": True, "scores_written": lock["written"], "results_written": time.strftime("%Y-%m-%d %H:%M"), "result_is": lock["result_is"], "test_run": TEST,
            "ground_truth": {"commit": EV_COMMIT, "sha256": GT_SHA}, "choice": lock["choice"], "systems": _sys}
    EV_RES.write_text(json.dumps(EVAL, indent=1))
    del GT

try:
    OFFICIAL = official_check()
except Exception as e:                                                # this notebook's numbers do not depend on this check
    OFFICIAL = {k: {"ran": False, "reason": f"an unexpected error ({repr(e)[:200]})"} for k in EV_SETS}
(EV_DIR / "official_evaluator_check.json").write_text(json.dumps(OFFICIAL, indent=1))


def final_numbers(k, official=None):
    # the rules: if the official evaluator's numbers differ from this notebook's, the evaluator's are the result
    o = OFFICIAL[k] if official is None else official
    if o.get("ran") and not o.get("agree"):
        return summary({t: o["per_section"][t] for t in TYPES_EVAL}), "the official evaluator (its numbers differ from this notebook's)"
    return ({kk: v for kk, v in EVAL["systems"][k].items() if kk != "system"},
            "this notebook (the official evaluator gives the same numbers)" if o.get("ran") else "this notebook (the official evaluator was not run)")


FINAL, FINAL_SOURCE = {}, {}
for k in EV_SETS:
    FINAL[k], FINAL_SOURCE[k] = final_numbers(k)
    o = OFFICIAL[k]
    print(f"the official evaluator on the {k} scores: " + (f"the same numbers: {o['agree']} (largest difference {o['max_abs_difference']:.1e} over "
          f"{o['values_compared']} values; its official score {o['omega']:.6f})" if o["ran"] else f"NOT run: {o['reason']}"))
    if o["ran"] and not o["agree"]:
        print("  the numbers differ: by the rules, the official evaluator's are the result")


def q1(omega):
    # the rules' word for the official score, in % of two decimals as the results page prints it, and its place beside the 35 teams
    a = round(100 * omega, 2)
    word = ("above the first team" if a > Q1_FIRST else "with the leading teams" if a >= Q1_LEADING else "in the upper half" if a >= Q1_UPPER
            else "above the baselines" if a >= Q1_BASE else "below the baselines")
    above = [k_ for k_, v in TEAMS.items() if round(v, 2) > a]
    return {"omega_percent": a, "word": word, "place_beside_the_35_teams": 1 + len(above), "teams_above": len(above), "next_team_above": above[-1] if above else None,
            "baselines_percent": {k_: round(v, 2) for k_, v in BASELINES.items()}, "first_team_percent": Q1_FIRST}


OTHER_IS = "kept" if RESULT_IS == "kept_AT" else "kept_AT"
K = FINAL[RESULT_IS]
Q1, Q1_OTHER = q1(K["omega"]), q1(FINAL[OTHER_IS]["omega"])
TRIO = {"as_it_is": {"ZP": FINAL["kept"], "frozen": FINAL["frozen"], "view_paper": FINAL["view"]},
        "attribute_classes": {"ZP": FINAL["kept_AT"], "frozen": FINAL["frozen_AT"], "view_paper": FINAL["view"]}}
Q2 = {k_: shift_table(v) for k_, v in TRIO.items()}
Q3 = {k_: fusion_table(v) for k_, v in TRIO.items()}
_lab = {"kept": "the system as it is", "kept_AT": "the system with attribute classes", "frozen": "frozen recipe, as it is", "frozen_AT": "frozen recipe, attribute classes",
        "view": "the view"}
print(f"\nEvaluation set, scored once ({EVAL['scores_written']}) · the result is {_lab[RESULT_IS]}:")
print(f"  {'':<36}{'Ω':>9}{'source':>9}{'target':>9}{'pAUC':>9}   " + "".join(f"{NAME[t][:10]:>11}" for t in TYPES_EVAL))
for k in ("kept", "kept_AT", "frozen", "frozen_AT", "view"):
    r = FINAL[k]
    print(f"  {_lab[k]:<36}{r['omega']:>9.4f}{r['AUC_source']:>9.4f}{r['AUC_target']:>9.4f}{r['pAUC']:>9.4f}   " + "".join(f"{r['per_type'][t]['score']:>11.4f}" for t in TYPES_EVAL))
print(f"\nThe result · {_lab[RESULT_IS]}: official score {Q1['omega_percent']:.2f} % → {Q1['word'].upper()} · place {Q1['place_beside_the_35_teams']} beside the 35 teams "
      f"(first team {Q1_FIRST}, third {Q1_LEADING}, eighteenth {Q1_UPPER}; baselines {Q1_BASE} and {round(BASELINES['DCASE2025_baseline_task2_MSE'], 2)})")
print(f"The other · {_lab[OTHER_IS]}: {Q1_OTHER['omega_percent']:.2f} % → {Q1_OTHER['word']} · place {Q1_OTHER['place_beside_the_35_teams']} (reported; the development set chose)")
print("The shift (harmonic mean AUC of the source domain, of the target domain, the difference):",
      {k_: {{"ZP": "system", "frozen": "frozen recipe", "view_paper": "view"}[nm]: (v["AUC_source"], v["AUC_target"], v["source_less_target"]) for nm, v in d.items()}
       for k_, d in Q2.items()},
      f"· the Mahalanobis baseline: {AE_EVAL_SHIFT[0] / 100:.4f} and {AE_EVAL_SHIFT[1] / 100:.4f}")
print("The fusion (the system's Ω less the better of its two parts):", {k_: (v["kept_less_better_part"], v["word"]) for k_, v in Q3.items()},
      "· on the development set, as it is:", (FUSION_DEV["kept_less_better_part"], FUSION_DEV["word"]))

In [ ]:
# 17 · Save, one picture, and one zip to download
import matplotlib.pyplot as plt

RESULT = {"rules": RULES, "date": time.strftime("%Y-%m-%d %H:%M"), "test_run": TEST,
          "meta": {"device": DEVICE, "gpu": GPU, "gpu_gb": round(GPU_GB, 1), "torch": torch.__version__, "torchaudio": torchaudio.__version__,
                   "clip_samples_kept": int(NS), "types": TYPES, "development_types": TYPES_DEV, "evaluation_types": TYPES_EVAL, "section": SECTION,
                   "frozen_recipe": {"cfg": CFG, "system": [[e, c, sd] for e, c, sd in FROZEN], "classes": F_CLASSES, "n_train": F_TRAIN,
                                     "clips_by_domain": F_ADD_DOMAINS, "encoders": ENC_INFO, "GB": GBR, "echo": INFO["ECHO"],
                                     "checkpoint_sha256": CKPT_SHA, "code_sha256": code_sha,
                                     "train_and_pass_seconds": T_TRAIN_F, "echo_seconds": T_ECHO, "part_1_seconds": T_PART1},
                   "attribute_classes": {"classes": AT_CLASSES, "classes_per_type": AT_PER_TYPE, "train_and_pass_seconds": T_TRAIN_AT},
                   "view": {"reference": {"repository": "https://github.com/soonhyeon/Noisy-ArcMix", "commit": REF_COMMIT, "licence": "MIT, Copyright (c) 2023 soonhyeon"},
                            "sv": SV, "n_frames": int(N_FRAMES), "last_kernel": list(LAST_KERNEL), "classes": CLASSES, "n_clips": int(len(ITEMS)),
                            "memory_layout_seconds_per_step": MF_TIMES, "train_seconds": T_TRAIN, "embed_seconds": T_EMBED, "part_2_seconds": T_PART2},
                   "zips": {"development": {k: list(v[1:]) for k, v in DC_DEV.items()}, "additional": {k: list(v[1:]) for k, v in XZIPS.items()},
                            "evaluation": {k: list(v[1:]) for k, v in DC_EVAL.items()}},
                   "evaluator": {"archive": EV_ARCHIVE[0], "commit": EV_COMMIT, "script": EV_SCRIPT, "tools": EV_TOOLS, "code_sha256": EV_SHA, "ground_truth_sha256": GT_SHA},
                   "prep_seconds": PREP_S, "session_minutes": round((time.time() - T_START) / 60, 1), "drive_free_gb": round(DRIVE_FREE_GB, 1)},
          "G0": {**G0, "c": G0C}, "G1": {"frozen_recipe": {"ok": DC_G1, "losses": G1_F}, "attribute_classes": {"ok": DC_G1_AT, "losses": G1_AT}, "view": G1},
          "G_DEV": {"ok": GDEV_OK, "systems": GDEV, "bar": GDEV_BAR},
          "counts": DCASE["counts"], "domains": DCASE["domains"], "clip_lengths_before_the_cut": DCASE["lengths"], "wav_formats": DCASE["formats"],
          "development": {"systems": DEV, "per_seed": PER_SEED, "shift": SHIFT_DEV, "fusion": FUSION_DEV, "published": PUB},
          "choice": CHOICE, "calibration": CAL, "evaluation": EVAL, "official_evaluator": OFFICIAL,
          "final": {"numbers": FINAL, "source": FINAL_SOURCE, "result_is": RESULT_IS},
          "readings": {"result": Q1, "other": Q1_OTHER, "shift": Q2, "fusion": Q3, "mahalanobis_baseline_evaluation_percent": list(AE_EVAL_SHIFT)},
          "yardsticks": {"teams_official_score_percent": TEAMS, "baselines_official_score_percent": BASELINES, "published_development_percent": PUBLISHED_DEV,
                         "lines_percent": {"first": Q1_FIRST, "leading": Q1_LEADING, "upper_half": Q1_UPPER, "baselines": Q1_BASE}, "fusion_bar": Q3_BAR},
          "training_logs": {"frozen_recipe": LOGS_F, "attribute_classes": LOGS_AT, "view": LOGS}}
(OUT / "first_shot.json").write_text(json.dumps(RESULT, indent=1, default=float, ensure_ascii=False))
# every development test clip's score under every system, with its label and domain, so that each development number above can be
# recomputed from the zip
SC_DIR = OUT / "development_scores"; SC_DIR.mkdir(exist_ok=True)
for (t, q), d in DEV_SC.items():
    (SC_DIR / f"scores_{t}_{q}.csv").write_text("file,label,domain," + ",".join(SYSTEMS) + "\n" + "".join(
        f"{d['file'][i]},{int(d['label'][i])},{d['domain'][i]}," + ",".join(repr(float(d[nm][i])) for nm in SYSTEMS) + "\n" for i in range(len(d["file"]))))
# the clips the two spreads rest on: the frozen recipe's score of the additional-training clips it did not train on, and the paper's
# score of the view's held-out clips. No test clip.
CAL_DIR = OUT / "calibration_scores"; CAL_DIR.mkdir(exist_ok=True)
for t in TYPES:
    (CAL_DIR / f"frozen_{t}.csv").write_text("file,frozen_left_out\n" + "".join(f"{a},{float(b)!r}\n" for a, b in CALF_SC[t]))
    (CAL_DIR / f"frozen_AT_{t}.csv").write_text("file,frozen_left_out\n" + "".join(f"{a},{float(b)!r}\n" for a, b in CALF_SC_AT[t]))
    _rows = np.asarray(HOLD_ROWS[t], dtype=np.int64)
    (CAL_DIR / f"held_out_{t}.csv").write_text("file,paper\n" + "".join(
        f"{DMETA[t]['name'][int(r)]},{float(a)!r}\n" for r, a in zip(_rows, paper(SEEDS, t, _rows))))

INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e4e3df"
COL = {"kept": "#1baf7a", "frozen": "#2a78d6", "view": "#eb6834", "kept_AT": "#8a5cd6"}   # four categorical slots (the first three: notebook 27's validated set)
MARK = {"kept": "D", "frozen": "o", "view": "s", "kept_AT": "^"}      # a shape per system as well as a colour
LAB4 = {"kept": "the system as it is: frozen + view", "frozen": "frozen recipe alone (as it is)", "view": "second view alone",
        "kept_AT": "with attribute classes: frozen (AT) + view"}


def note(ax, x_, y_, s_, below=False):
    ax.annotate(s_, (x_, y_), xytext=(0, -9 if below else 9), textcoords="offset points", ha="center", va="top" if below else "bottom", fontsize=8, color=INK,
                zorder=5, bbox=dict(facecolor="white", edgecolor="none", pad=0.5, alpha=0.9))


fig, axes = plt.subplots(1, 2, figsize=(15.5, 6.6), gridspec_kw={"width_ratios": [1.75, 1.0]})
ax = axes[0]; x = np.arange(len(TYPES_EVAL) + 1); order = ("frozen", "view", "kept", "kept_AT"); w = 0.17
for j, k in enumerate(order):
    vals = [FINAL[k]["per_type"][t]["score"] for t in TYPES_EVAL] + [FINAL[k]["omega"]]
    ax.plot(x + (j - 1.5) * w, vals, MARK[k], ms=9.5 if k == RESULT_IS else 7.5, color=COL[k], mec=INK if k == RESULT_IS else "white", mew=1.2,
            label=LAB4[k] + (" (the result)" if k == RESULT_IS else ""), zorder=4)
    if k in ("kept", "kept_AT"):
        note(ax, x[-1] + (j - 1.5) * w, vals[-1], f"{vals[-1]:.4f}", below=(k == "kept"))
ax.hlines([v / 100 for v in TEAMS.values()], x[-1] + 0.38, x[-1] + 0.48, color=INK, lw=1.1, zorder=3, label="the challenge's 35 teams (each team's best system)")
ax.hlines([v / 100 for v in BASELINES.values()], x[-1] - 0.48, x[-1] + 0.48, color=MUTED, lw=1.1, ls=(0, (4, 2)), zorder=2, label="the challenge's two baselines")
ax.axvline(x[-1] - 0.5, color=GRID, lw=1.0)
ax.set_xticks(x, [NAME[t] for t in TYPES_EVAL] + ["all types\n(official score Ω)"], fontsize=8.5); ax.set_xlim(-0.6, len(TYPES_EVAL) + 0.6)
_lo = max(0.0, min(0.4, math.floor(min(FINAL[k]["per_type"][t]["score"] for k in order for t in TYPES_EVAL) * 20) / 20 - 0.05)); ax.set_ylim(_lo, 1.0)
ax.set_ylabel("harmonic mean of a type's three values (AUC source, AUC target, pAUC)", color=MUTED, fontsize=8.5)
ax.set_title(f"Evaluation set, scored once ({EVAL['scores_written']})" + ("; the scale does not start at zero" if _lo > 0 else ""), loc="left", color=INK, fontsize=10.5)
ax.legend(frameon=False, fontsize=8.5, loc="upper center", bbox_to_anchor=(0.5, -0.11), ncol=2)
ax = axes[1]
groups = [("development", {"frozen": DEV["frozen"], "view": DEV["view_paper"], "kept": DEV["ZP"], "kept_AT": DEV["ZP_AT"]}, PUB["Autoencoder baseline (Mahalanobis)"]),
          ("evaluation", {k: FINAL[k] for k in order}, {"AUC_source": AE_EVAL_SHIFT[0] / 100, "AUC_target": AE_EVAL_SHIFT[1] / 100})]
pos, ticks, _all = 0.0, [], []
for g, (gname, res, ae) in enumerate(groups):
    for k in order + ("AE",):
        a, b = (ae["AUC_source"], ae["AUC_target"]) if k == "AE" else (res[k]["AUC_source"], res[k]["AUC_target"])
        c = MUTED if k == "AE" else COL[k]
        ax.plot([pos, pos], [a, b], color=c, lw=2, zorder=2)
        ax.plot(pos, a, "o", ms=8.5, color=c, mec="white", mew=1.2, zorder=4, label="source domain (filled)" if (g, k) == (0, "frozen") else None)
        ax.plot(pos, b, "o", ms=8.5, color="white", mec=c, mew=2, zorder=4, label="target domain (open)" if (g, k) == (0, "frozen") else None)
        note(ax, pos, max(a, b), f"{max(a, b):.3f}"); note(ax, pos, min(a, b), f"{min(a, b):.3f}", below=True); _all += [a, b]
        ticks.append((pos, {"frozen": "frozen", "view": "view", "kept": "as it is", "kept_AT": "AT", "AE": "base-\nline"}[k])); pos += 1.0
    ax.annotate(gname + " set", (pos - 3.0, 1.0), xycoords=("data", "axes fraction"), xytext=(0, 3), textcoords="offset points", ha="center", va="bottom", fontsize=9.5, color=INK)
    pos += 0.6
ax.set_xticks([p for p, _ in ticks], [s for _, s in ticks], fontsize=8); ax.set_xlim(-0.7, pos - 0.9)
_lo2 = max(0.0, min(0.3, math.floor(min(_all) * 10) / 10 - 0.1)); ax.set_ylim(_lo2, 1.0)
ax.set_ylabel("harmonic mean AUC over the types" + ("; the scale does not start at zero" if _lo2 > 0 else ""), color=MUTED, fontsize=8.5)
ax.set_title("Source conditions against target conditions\n", loc="left", color=INK, fontsize=10.5)
ax.legend(frameon=False, fontsize=8.5, loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=2)
for ax in axes:
    ax.grid(axis="y", color=GRID, lw=0.6); ax.set_axisbelow(True); ax.tick_params(colors=MUTED)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
fig.suptitle("P04 · notebook 30 · the kept system on DCASE 2025 Task 2 (first-shot: eight machine types met with one section each)\n"
             f"the result ({'attribute classes' if RESULT_IS == 'kept_AT' else 'as it is'}): official score {Q1['omega_percent']:.2f} % → {Q1['word']} · "
             f"place {Q1['place_beside_the_35_teams']} beside the 35 teams · the other system: {Q1_OTHER['omega_percent']:.2f} %", fontsize=11, x=0.012, y=0.985, ha="left")
fig.subplots_adjust(left=0.05, right=0.99, top=0.83, bottom=0.2, wspace=0.12); fig.savefig(OUT / "first_shot.png", dpi=150); plt.show()

ZIP = OUT / "p04_n30_results.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for f in ("first_shot.json", "first_shot.png"):
        z.write(OUT / f, f)
    for d in (SC_DIR, CAL_DIR):
        for f in sorted(d.glob("*.csv")):
            z.write(f, f"{d.name}/{f.name}")
    for f in sorted(p for p in EV_DIR.rglob("*") if p.is_file()):
        z.write(f, f"{EV_DIR.name}/{f.relative_to(EV_DIR).as_posix()}")
print(f"saved {ZIP} · {(time.time() - T_START) / 60:.0f} min in this session")
if IN_COLAB:
    from google.colab import files
    files.download(str(ZIP))

## Send back
Send the zip that downloaded (`p04_n30_results.zip`). It is also in Drive › `P04_FirstShot`.